# 🤖 13장 · 몸을 가진 AI: 센서 데이터로 읽는 피지컬 AI (실습 노트북 v6)

『빅데이터분석 with 파이썬』 13장의 **05. 파이썬 실습**과 **종합 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 13-1, 13-2, 13-3, 13-4, 13-5, 13-6 |
| ☆ 선택 | 자습이나 심화 | 13-7, 13-8, 13-9, 13-10, 13-11 |

**데이터**
- 동작 인식: UCI "Human Activity Recognition Using Smartphones" (30명, 2.56초 윈도우 10,299개). GitHub 저장소 `bidhanbaray/UCI_HAR_Dataset`에서 필요한 파일만 내려받습니다(약 200MB).
- 공장 기계: UCI "AI4I 2020 Predictive Maintenance Dataset" (작업 10,000건, 합성 데이터). GitHub 저장소 `lingchenlijing/AI4I-2020-Predictive-Maintenance-Dataset`에서 바로 읽습니다.
- 종합 프로젝트: 한국 공공데이터 파일을 직접 올려서 씁니다(맨 아래 안내).

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

**v5(2026-11-11)**: 책 v8 원고와 같은 차례와 이름([실습 예제 13-k], 단계 ❶❷)입니다. 알고리즘이 처음 나올 때마다 [알고리즘 따라 하기]에서 작은 예제로 먼저 계산을 따라 하고, 그다음 실제 데이터 실습을 합니다. 끝에 책에서 뺀 코랩 실습 13-7~13-11이 있습니다.

> 📒 **코랩 실습 표시**: 제목에 **[코랩 실습 13-k]**가 붙은 칸(13-5, 13-6 ❻, 13-7~13-11)은 책에 코드가 없고 이 노트북에만 있습니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**를 설치합니다. 판다스, 사이킷런, 맷플롯립은 코랩에 이미 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch13/data/"
for f in [
    "asos_서울108_시간자료_2023_2026.csv",
    "kpx_시간별_전국_전력수요량_2021_2025.csv",
    "서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv",
    "에어코리아_서울_도시대기평균_시간별_2023_2026.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 허리의 휴대폰에서 공장의 기계까지

03절의 분석을 코드로 처음부터 끝까지 따라 합니다. 코랩에서 노트북 "13장_피지컬AI_실습_colab_v6.ipynb"를 열고 위에서부터 차례로 실행하면 책과 같은 숫자가 나옵니다. ★ 실습과 [알고리즘 따라 하기]는 코드를 한 줄도 빼지 않고 책에 실었습니다. 노트북은 같은 코드를 셀로 나누고, 코랩 실습 13-7~13-11(기울기 각도, 특징 561개, 문턱값 고르기, 이상 탐지, 모델 크기)을 더한 것입니다.

> **[STEP] 실습 전에 미리 살펴보기**
>
> 데이터: 동작 인식(HAR)과 공장 기계(AI4I)의 깃허브 사본⁹⁾. 인터넷만 되면 코드가 알아서 내려받습니다.
>
> 도구: 넘파이(신호 배열), 판다스(표), 사이킷런(모델), 맷플롯립(그래프). 코랩에는 모두 깔려 있습니다.
>
> 순서: 13-1~13-5는 동작 인식, 13-6은 공장 데이터로 03절을 그대로 따라갑니다. FFT, 결정나무, 랜덤 포레스트, 정밀도·재현율이 처음 나오는 자리에는 [알고리즘 따라 하기] 13-1~13-4에서 숫자 8~10개짜리 작은 예제로 계산을 한 단계씩 먼저 따라 한 뒤 실제 데이터로 넘어갑니다. 코랩 실습 13-7~13-11은 코랩 노트북에만 있습니다.
>
> 읽는 법: 단계(①②③…)마다 **할 일 → 코드 → 줄별 설명 → 실행 결과 → 결과 읽기** 순서입니다. 코드를 직접 치기 전에 줄별 설명을 먼저 읽고, 실행한 뒤에는 결과 읽기와 내 화면의 숫자를 맞대 보십시오.
>
> 개념 상자: 처음 나오는 데이터 분석·머신러닝 기법은 [잠깐만요] 상자로, 처음 나오는 파이썬 문법은 [코드 읽기] 상자로 그 자리에서 풀었습니다.

> **[잠깐만요] 실습이 막힐 때 확인할 세 가지**
>
> ① **위 셀부터 실행했나?** 노트북은 위에서 만든 변수(`sig_tr`, `F_tr`, `df` 등)를 아래에서 씁니다. `NameError: name 'F_tr' is not defined`가 나오면 그 변수를 만드는 위 셀을 먼저 실행합니다.
>
> ② **모양(shape)이 예상과 같은가?** 결과가 이상하면 `print(배열.shape)`, `print(표.head( ))`로 중간 결과부터 찍어 봅니다. 대부분의 실수는 축이나 열을 잘못 고른 것입니다.
>
> ③ **숫자가 책과 조금 다른가?** 사이킷런 버전에 따라 소수 둘째 자리가 달라질 수 있습니다. 비교의 방향(어느 쪽이 높은가)이 같으면 괜찮습니다.

#### [실습 예제 13-1 ★] 동작 인식 데이터 불러오고 누가 만들었는지 보기

HAR 원신호 9개 채널을 내려받아 (윈도우 수, 시간 128칸, 채널 9) 모양의 배열로 불러오고, 학습·시험에 나온 사람과 한 사람이 남긴 윈도우 수를 세어 그래프로 그리시오(03절 3.1).

**❶ 도구 불러오고 데이터 파일 내려받기**

넘파이·판다스·맷플롯립을 불러오고 한글 글꼴을 맞춘 뒤, 필요한 파일 22개만 깃허브 사본에서 내려받습니다.

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib            # 코랩: 한글 글꼴 자동 설정
except ImportError:                        # 나눔고딕이 이미 있는 컴퓨터
    import glob
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

HAR = "UCI HAR Dataset"
MIRROR = ("https://raw.githubusercontent.com/"
          "bidhanbaray/UCI_HAR_Dataset/master/")
CH = [f"{s}_{a}" for s in ["body_acc", "body_gyro", "total_acc"]
      for a in "xyz"]                      # 채널 9개의 이름
files = []
for s in ["train", "test"]:
    files += [f"{s}/y_{s}.txt", f"{s}/subject_{s}.txt"]
    files += [f"{s}/Inertial Signals/{c}_{s}.txt" for c in CH]
for f in files:                            # 없는 파일만 내려받는다
    path = f"{HAR}/{f}"
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(MIRROR + f.replace(" ", "%20"), path)
print("파일", len(files), "개 준비 끝")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `try: … except ImportError:` 코랩에서는 `pip install koreanize-matplotlib` 한 번이면 한글이 나옵니다. 그 꾸러미가 없으면 컴퓨터에 있는 나눔고딕을 찾아 등록합니다. `axes.unicode_minus = False`는 음수 부호가 네모로 깨지는 것을 막습니다.
>
> `CH = [… for s in … for a in "xyz"]` 두 겹 리스트 만들기입니다. 바깥 `s`(신호 종류 3개)마다 안쪽 `a`(축 3개)를 돌아 `body_acc_x`, `body_acc_y`, …, `total_acc_z`의 9개 이름을 만듭니다.
>
> `files += […]` 학습(train)·시험(test)마다 정답(`y_`), 사람 번호(`subject_`), 채널 9개, 모두 11개씩 22개 파일의 경로를 모읍니다.
>
> `os.path.exists(path)` 이미 받은 파일은 건너뜁니다. 셀을 다시 실행해도 또 받지 않습니다.
>
> `f.replace(" ", "%20")` 폴더 이름 "Inertial Signals"의 빈칸을 인터넷 주소에서 쓰는 `%20`으로 바꿉니다.



원래 데이터는 UCI 저장소에서 받지만, 이 책은 같은 파일을 올려 둔 깃허브 사본을 씁니다. 연구팀이 만든 특징 561개 파일(약 92MB)은 지금 받지 않고, 코랩 실습 13-8에서 필요할 때 받습니다.

**❷ 파일 한 줄 들여다보기**

배열로 만들기 전에, 파일 하나가 어떻게 생겼는지 첫 줄만 읽어 봅니다.

In [ ]:
f = f"{HAR}/train/Inertial Signals/total_acc_x_train.txt"
row = np.loadtxt(f, max_rows=1)            # 첫 줄 하나만 읽기
print("첫 줄의 숫자 개수:", row.size)
print("앞의 5개:", row[:5].round(3))
print("정답 앞 5줄:", open(f"{HAR}/train/y_train.txt").read().split()[:5])
print("사람 앞 5줄:", open(f"{HAR}/train/subject_train.txt").read().split()[:5])

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.loadtxt(f, max_rows=1)` 빈칸으로 구분된 숫자 파일을 읽습니다. `max_rows=1`이면 첫 줄만 읽어 빠릅니다.
>
> `row[:5].round(3)` 앞의 5개를 소수 셋째 자리로 반올림해 봅니다.
>
> `open(…).read().split()[:5]` 정답 파일과 사람 파일은 한 줄에 숫자가 하나뿐이라, 글자로 읽어 빈칸·줄바꿈으로 쪼갠 뒤 앞 5개만 봅니다.



한 줄이 윈도우 하나(128칸 = 2.56초)입니다. 전체 가속도 x축이 약 1g이니 휴대폰이 서 있는 자세입니다. 같은 줄 번호끼리 짝이 맞습니다. 첫 윈도우는 1번 사람이 한 5번 동작(서기)입니다. 파일 셋이 줄 번호로만 이어져 있으니, 줄을 섞거나 빼면 짝이 틀어집니다.

**❸ 채널 9장을 3차원 배열로 쌓기**

In [ ]:
def load(split):                           # (윈도우, 시간 128칸, 채널 9)
    sig = np.stack([np.loadtxt(f"{HAR}/{split}/Inertial Signals/"
                               f"{c}_{split}.txt") for c in CH], axis=2)
    y = np.loadtxt(f"{HAR}/{split}/y_{split}.txt").astype(int) - 1
    s = np.loadtxt(f"{HAR}/{split}/subject_{split}.txt").astype(int)
    return sig, y, s

sig_tr, y_tr, s_tr = load("train")
sig_te, y_te, s_te = load("test")
print("학습:", sig_tr.shape, " 시험:", sig_te.shape)
print("한 윈도우 =", sig_tr.shape[1] / 50, "초 (50Hz)")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `def load(split):` 학습과 시험을 똑같이 읽으므로 함수로 만들어 두 번 부릅니다.
>
> `np.stack([…], axis=2)` (윈도우, 128) 표 9장을 세 번째 방향으로 쌓아 (윈도우, 128, 9)를 만듭니다. 아래 [코드 읽기] 상자를 보십시오.
>
> `.astype(int) - 1` 정답 1~6을 정수로 바꾸고 1을 빼서 0~5로 맞춥니다. 파이썬 목록의 번호가 0부터 시작하기 때문입니다.
>
> `s` 사람 번호입니다. 3.4에서 사람 단위로 나눌 때 이 열이 없으면 아무것도 할 수 없습니다. 데이터를 받으면 "주인을 가리키는 열"부터 챙깁니다.
>
> `sig_tr.shape[1] / 50` 시간 칸 수(128)를 샘플링 주파수(50Hz)로 나누면 윈도우 길이(초)입니다.



학습 7,352개와 시험 2,947개, 모두 10,299개 윈도우입니다. 숫자는 10,299 × 128 × 9 ≈ 1,190만 개입니다.

> **[코드 읽기] `np.stack( …, axis=2)`**
>
> `np.loadtxt( )`는 채널 하나를 (윈도우 수, 128) 표로 읽습니다. `np.stack( )`은 같은 모양의 표 여러 장을 겹쳐 차원을 하나 늘립니다. `axis=0`이면 맨 앞에 쌓아 (9, 윈도우, 128), `axis=2`면 맨 뒤에 쌓아 (윈도우, 128, 9)가 됩니다. "윈도우 하나 = 128칸 × 9채널"로 생각하기 쉽도록 채널을 맨 뒤에 두었습니다. 비슷한 `np.concatenate( )`는 차원을 늘리지 않고 이어 붙입니다(다음 단계의 `np.r_`과 같습니다).

**❹ 누가 학습용, 누가 시험용인지 세고 그리기**

In [ ]:
ACT = ["걷기", "계단 오르기", "계단 내려가기", "앉기", "서기", "눕기"]
y_all, s_all = np.r_[y_tr, y_te], np.r_[s_tr, s_te]
print("학습에 나온 사람:", sorted(map(int, set(s_tr))))
print("시험에 나온 사람:", sorted(map(int, set(s_te))))
print("겹치는 사람:", set(s_tr) & set(s_te))
per = pd.Series(s_all).value_counts().sort_index()
print("사람 수:", len(per), " 한 사람의 윈도우 수:", per.min(), "~", per.max())
cnt = pd.Series(y_all).map(dict(enumerate(ACT))).value_counts()
print(cnt.reindex(ACT).to_string())

color = ["#C8234B" if s in set(s_te) else "#9a9a9a" for s in per.index]
per.plot.bar(color=color, figsize=(7.5, 3))
plt.xlabel("사람 번호 (빨강: 시험용 9명, 회색: 학습용 21명)")
plt.ylabel("윈도우 수")
plt.xticks(rotation=0, fontsize=10)
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.r_[A, B]` 두 배열을 앞뒤로 잇습니다. 학습과 시험을 합친 전체(10,299개)를 따로 둡니다.
>
> `set(s_tr)` 겹치는 값을 하나만 남겨 사람 번호의 모음을 만듭니다. `&`는 두 모음에 모두 있는 것(교집합)입니다.
>
> `value_counts( ).sort_index( )` 사람 번호마다 몇 번 나왔는지(=윈도우 수) 세고, 번호 순으로 줄을 세웁니다.
>
> `.map(dict(enumerate(ACT)))` 0~5 숫자를 {0: "걷기", …} 사전으로 동작 이름으로 바꿉니다. `reindex(ACT)`는 책의 동작 순서대로 줄을 세웁니다.
>
> `color = [… if … else …]` 시험용 9명이면 빨강, 아니면 회색인 색 목록입니다. `per.plot.bar( )`는 번호(index)를 가로축, 값을 막대 높이로 그립니다.



*(그래프: 사람별 윈도우 수. 빨간 막대 9명은 시험에만, 회색 막대 21명은 학습에만 나온다. 한 사람이 윈도우를 281~409개 남겼다.)*

`set()`은 빈 모음, 곧 학습과 시험에 겹치는 사람이 없다는 뜻입니다. 연구팀이 사람 단위로 나누어 두었습니다. 한 사람이 윈도우를 300개 넘게 남기므로, 줄 단위로 섞으면 같은 사람이 학습과 시험 양쪽에 들어갑니다. 여섯 동작의 개수는 1,406~1,944개로 비슷해서 정확도 하나로 성적을 보아도 크게 무리가 없습니다(고장 3.39%인 13-6과 비교해 보십시오). 나이와 성별 칸은 없습니다. 누가 빠졌는지는 데이터가 아니라 논문을 읽어야 압니다.

> **[잠깐만요] 배열의 모양(shape)과 축(axis)**
>
> 넘파이 배열은 표를 겹겹이 쌓은 것입니다. `sig_tr.shape`가 (7352, 128, 9)이면 0번 축은 윈도우 7,352개, 1번 축은 시간 128칸, 2번 축은 채널 9개입니다.
>
> 함수에 `axis=`를 주면 그 축 방향으로 계산하고, 그 축은 사라집니다. `sig_tr.mean(axis=1)`은 시간 방향 평균이라 결과가 (7352, 9)가 됩니다. 엑셀 표로 치면 `axis=0`은 열마다 위아래로, `axis=1`은 행마다 좌우로 계산하는 것입니다.
>
> 다른 데이터도 같습니다. 컬러 사진 한 장은 (세로, 가로, 3), 사진 100장은 (100, 세로, 가로, 3)입니다. 코드가 헷갈리면 먼저 `.shape`부터 찍어 보세요.

#### [알고리즘 따라 하기 13-1] FFT를 숫자 8개로 직접 계산하기

[실습 예제 13-2]에서 쓸 FFT(고속 푸리에 변환)가 무엇을 계산하는지, 숫자 8개짜리 신호로 먼저 손으로 따라 해 봅니다. 1초에 8번 잰 신호에 2Hz 물결과 그 절반 세기의 3Hz 물결을 섞어 두고, FFT가 이 둘을 찾아내는지 봅니다.

**❶ 신호 만들기: 2Hz와 3Hz 물결 섞기**

1초에 8번(fₛ = 8Hz) 재는 1초 분량의 신호를 만듭니다. 답을 미리 알고 시작하는 셈입니다.

In [ ]:
import numpy as np
fs = 8                                    # 1초에 8번 잰다
t = np.arange(8) / fs                     # 0, 0.125, …, 0.875초
x = np.sin(2 * np.pi * 2 * t) + 0.5 * np.sin(2 * np.pi * 3 * t)
print("시간:", t)
print("신호:", x.round(2))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `t = np.arange(8) / fs` 0, 1/8, 2/8, … 초. 8칸의 시각입니다.
>
> `np.sin(2 * np.pi * 2 * t)` 1초에 2번 오르내리는 물결(2Hz)입니다. `2 * np.pi`는 한 바퀴(360°)입니다.
>
> `+ 0.5 * np.sin(… 3 * t)` 세기가 절반인 3Hz 물결을 더합니다. 실제 걷기 신호도 이렇게 여러 물결의 합입니다.



숫자 8개만 보아서는 어떤 물결이 섞였는지 알 수 없습니다. 이것을 알아내는 것이 FFT입니다.

**❷ 칸마다 "이 빠르기의 물결과 얼마나 닮았나" 재기**

0Hz, 1Hz, …, 4Hz 칸마다 그 빠르기의 코사인·사인 물결을 만들어, 신호와 곱해 더합니다. 닮았으면 큰 값, 닮지 않았으면 0이 나옵니다.

In [ ]:
N = len(x)
for k in range(N // 2 + 1):               # 0, 1, 2, 3, 4Hz 칸
    c = np.cos(2 * np.pi * k * np.arange(N) / N)
    s = np.sin(2 * np.pi * k * np.arange(N) / N)
    re, im = round((x * c).sum(), 2) + 0.0, round(-(x * s).sum(), 2) + 0.0
    print(f"{k}Hz 칸: 코사인 닮음 {re:5.2f}, 사인 닮음 {im:5.2f}, 크기 {np.hypot(re, im):4.2f}")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `range(N // 2 + 1)` 8칸이면 0~4Hz, 5개 칸입니다. 4Hz(= 8 ÷ 2)가 볼 수 있는 가장 빠른 흔들림입니다(식 13-3).
>
> `(x * c).sum()` 신호와 코사인 물결을 칸마다 곱해 더한 값입니다. 둘이 같이 오르내리면 커지고, 엇갈리면 0에 가깝습니다.
>
> `np.hypot(re, im)` √(re² + im²). 코사인과 사인 두 쪽의 닮음을 합친 크기입니다. 물결이 어느 순간에 시작했든 같은 크기가 나옵니다.



2Hz 칸이 4.00, 3Hz 칸이 2.00이고 나머지는 0입니다. 섞어 넣은 두 물결과 세기의 비(1 : 0.5)를 그대로 찾아냈습니다. 크기 4는 칸 수의 절반(8 ÷ 2)에 세기 1을 곱한 값입니다.

**❸ np.fft.rfft로 한 번에 계산하고 가장 센 칸 읽기**

같은 계산을 넘파이의 `rfft`로 한 줄에 하고, 가장 센 칸을 Hz로 바꿉니다. [실습 예제 13-2]의 걸음 리듬 찾기와 같은 순서입니다.

In [ ]:
amp = np.abs(np.fft.rfft(x))
print("rfft 크기:", amp.round(2))
print("칸 간격:", fs / N, "Hz,  가장 센 칸:", amp.argmax(), "→", amp.argmax() * fs / N, "Hz")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.abs(np.fft.rfft(x))` 앞 단계의 다섯 칸 크기를 한 번에 구합니다. 칸 수가 128이어도 빠르게 계산합니다.
>
> `fs / N` 칸 간격 Δf = fₛ ÷ N = 8 ÷ 8 = 1Hz입니다(식 13-4).
>
> `amp.argmax( ) * fs / N` 가장 센 칸의 번호에 칸 간격을 곱하면 그 칸의 빠르기(Hz)입니다.



손으로 계산한 값과 `rfft`가 똑같습니다. 가장 센 칸은 2Hz입니다. 실제 데이터에서는 fₛ = 50Hz, N = 128이라 칸 간격이 0.39Hz가 될 뿐 읽는 법은 같습니다.

#### [실습 예제 13-2 ★] 중력이 알려 주는 자세, FFT가 알려 주는 리듬

동작마다 (1) 세 축의 평균 가속도(자세), (2) 흔들림의 크기, (3) 가장 센 주파수(리듬)를 구하고 그래프로 견주시오(03절 3.2).

**❶ 동작별 축 평균 = 자세**

In [ ]:
sig_all = np.r_[sig_tr, sig_te]
grav = pd.DataFrame(
    [sig_all[y_all == i][:, :, 6:9].mean(axis=(0, 1)) for i in range(6)],
    index=ACT, columns=["x", "y", "z"])     # 동작별 축 평균(g)
print(grav.round(3))

grav.plot.bar(color=["#C8234B", "#9a9a9a", "#4d4d4d"], figsize=(6.5, 3.2))
plt.xticks(range(6), ["걷기", "오르기", "내려가기", "앉기", "서기", "눕기"],
           rotation=0)
plt.ylabel("평균 가속도 (g)")
plt.legend(title="축", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `sig_all[y_all == i]` 정답이 i인 윈도우만 고릅니다(참·거짓으로 골라내기, 아래 [잠깐만요] 상자).
>
> `[:, :, 6:9]` 채널 6, 7, 8번, 곧 전체 가속도(`total_acc`) x, y, z입니다. 중력이 들어 있는 채널입니다. `6:9`는 6부터 9 앞(8)까지입니다.
>
> `.mean(axis=(0, 1))` 윈도우 방향(0)과 시간 방향(1)으로 모두 평균해 축마다 숫자 하나를 남깁니다. 결과는 (3,)입니다.
>
> `pd.DataFrame([…], index=ACT, columns=[…])` 동작 6개 × 축 3개 표를 만듭니다. 표에 `.plot.bar( )`를 붙이면 열마다 색이 다른 묶음 막대가 됩니다.
>
> `bbox_to_anchor=(1.01, 1)` 범례를 그래프 오른쪽 바깥에 둡니다.



*(그래프: 동작별 축마다의 평균 가속도. 눕기만 x축(빨강)의 1g가 사라지고 y축, z축으로 옮겨 간다.)*

다섯 동작은 x축이 약 1g, 눕기만 x축이 0.069g입니다. 휴대폰을 허리에 세로로 찼으니, 서 있으면 휴대폰의 위아래(x축)가 중력과 나란하고 누우면 직각이 됩니다. 평균 하나로 눕기를 가려낼 수 있습니다. 앉기와 서기는 y·z축 부호만 조금 다릅니다(앉기 0.129·0.147, 서기 -0.165·-0.019). 이 작은 차이가 4.1의 기울기 각도와 코랩 실습 13-7로 이어집니다.

**❷ 흔들림의 크기 (식 13-1)**

In [ ]:
mag = np.linalg.norm(sig_all[:, :, 6:9], axis=2)    # (식 13-1) 크기
print("크기 배열:", mag.shape)
body = np.linalg.norm(sig_all[:, :, 0:3], axis=2).std(axis=1)
shake = pd.Series([np.median(body[y_all == i]) for i in range(6)], index=ACT)
print("흔들림(몸 가속도 크기의 표준편차, 중앙값)")
print(shake.round(3).to_string())

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.linalg.norm(…, axis=2)` 축 방향(2번 축, x·y·z)으로 √(x² + y² + z²)를 계산합니다. (식 13-1)을 한 줄로 쓴 것입니다. 축이 사라져 (10299, 128)이 됩니다.
>
> `mag` 전체 가속도의 크기입니다. 휴대폰이 어느 쪽을 향하든 같은 값이라 리듬(③)과 특징(13-3)에 씁니다.
>
> `sig_all[:, :, 0:3]` 몸 가속도(`body_acc`)는 중력을 걸러 낸 움직임만의 가속도입니다. 그 크기의 `.std(axis=1)`은 윈도우마다 시간 방향 표준편차, 곧 흔들림의 크기입니다.
>
> `np.median( )` 동작마다 윈도우 수백 개의 흔들림 가운데 가운데 값입니다. 유난히 센 윈도우 몇 개에 끌려가지 않도록 평균 대신 중앙값을 씁니다.



움직이는 세 동작(0.121~0.208)과 멈춘 세 동작(0.005~0.006)이 20배 넘게 차이 납니다. 숫자 하나로 두 무리가 갈리니, 13-3의 "질문 하나짜리 나무"가 이 숫자를 고르리라 짐작할 수 있습니다. 계단을 내려갈 때 가장 세게 흔들린다는 몸의 경험과도 맞습니다.

**❸ FFT로 걸음 리듬 찾기**

In [ ]:
fr = np.fft.rfftfreq(128, 1 / 50)          # 0, 0.39, 0.78, ... 25Hz
def spec(x):                               # 윈도우마다 주파수별 세기
    x = x - x.mean(axis=1, keepdims=True)  # 평균(중력)을 빼고
    return np.abs(np.fft.rfft(x * np.hanning(128), axis=1))

S = spec(mag)
dom = fr[1:][S[:, 1:].argmax(axis=1)]      # 0Hz 칸을 뺀 가장 센 주파수
walk = np.median(dom[y_all == 0])
print("칸 수:", fr.size, " 칸 간격:", round(fr[1], 3), "Hz",
      " 가장 높은 주파수:", fr[-1], "Hz")
print("걷기의 가장 센 주파수(중앙값):", round(walk, 2), "Hz →",
      "1분에", round(walk * 60), "걸음")

for i, c in [(0, "#C8234B"), (3, "#7a7a7a")]:
    plt.plot(fr, S[y_all == i].mean(axis=0), color=c, label=ACT[i])
plt.xlim(0, 10)
plt.xlabel("주파수 (Hz, 1초에 몇 번 흔들리나)")
plt.ylabel("세기 (평균)")
plt.legend()
plt.gcf().set_size_inches(6, 3)
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.fft.rfftfreq(128, 1 / 50)` 128칸을 0.02초(1/50초) 간격으로 잰 신호의 FFT 칸들이 각각 몇 Hz인지 알려 줍니다.
>
> `x - x.mean(axis=1, keepdims=True)` 윈도우마다 평균을 뺍니다. 평균(약 1g의 중력)이 그대로 있으면 0Hz 칸이 너무 커서 다른 칸이 묻힙니다. `keepdims=True`는 (윈도우, 1) 모양을 남겨 128칸 모두에서 뺄 수 있게 합니다.
>
> `np.fft.rfft(…, axis=1)`, `np.abs( )`, `np.hanning(128)` 아래 [코드 읽기] 상자를 보십시오.
>
> `S[:, 1:].argmax(axis=1)` 0Hz 칸을 뺀 나머지에서 가장 큰 칸의 번호를 윈도우마다 찾습니다. `fr[1:][번호]`로 그 칸의 주파수를 꺼냅니다.
>
> `S[y_all == i].mean(axis=0)` 동작 i의 윈도우들을 칸마다 평균한 "평균 스펙트럼"입니다.



*(그래프: 걷기(빨강)와 앉기(회색)의 평균 주파수 성분. 걷기는 2Hz 가까이에서 가장 세다. 1초에 약 두 걸음이다.)*

칸 65개, 간격 0.391Hz, 가장 높은 주파수 25Hz는 4.2의 (식 13-3)·(식 13-4)와 같습니다. 걷기는 1.95Hz(1초에 두 걸음쯤)에서 봉우리가 서고, 앉기는 거의 평평합니다. 117걸음은 [예제 13-13]의 답이기도 합니다. 칸 간격이 0.39Hz이므로 실제 리듬은 1.76~2.15Hz 사이 어딘가로 읽어야 합니다.

> **[코드 읽기] `np.fft.rfft( )`, `np.abs( )`, `np.hanning( )`**
>
> `np.fft.rfft(x, axis=1)`은 행마다(윈도우마다) FFT를 해서, 128칸 신호를 주파수 칸 65개(0~25Hz)로 바꿉니다. 결과는 크기와 위상(물결이 어디서 시작하는지)을 함께 담은 복소수라서, `np.abs( )`로 크기만 꺼내 "그 빠르기가 얼마나 센가"로 씁니다. `np.hanning(128)`은 가운데가 1이고 양 끝이 0인 종 모양 숫자 128개입니다. 신호에 곱하면 윈도우 양 끝이 부드럽게 0으로 줄어, 잘린 자리에서 생기는 가짜 주파수가 줄어듭니다. 이름의 r은 real(실수)로, 실수 신호에서는 절반(65칸)만 계산해도 충분하다는 뜻입니다.

> **[잠깐만요] 참·거짓으로 골라내기(불리언 인덱싱)**
>
> `y_all == 0`은 숫자 대신 참(True)·거짓(False)이 10,299개 담긴 배열을 돌려줍니다. 이것을 대괄호에 넣은 `sig_all[y_all == 0]`은 참인 자리의 윈도우만 남깁니다. 반복문 없이 원하는 줄을 한 번에 고르는 방법입니다.
>
> 조건 두 개는 `&`(그리고), `|`(또는)로 잇고 조건마다 괄호를 칩니다. 예: `(df.dT < 8.6) & (rpm < 1380)`. 판다스 표도 같습니다. 집값 표에서 서울의 30평 이상만 보려면 `df[(df["시도"] == "서울") & (df["평"] >= 30)]`이라고 씁니다.
>
> 참·거짓 배열은 숫자처럼 더하고 평균낼 수도 있습니다. 참은 1, 거짓은 0이므로 `(예측 == 정답).mean( )`이 곧 정확도입니다.

#### [알고리즘 따라 하기 13-2] 질문 하나짜리 나무를 손으로 고르기

결정나무는 "어떤 특징이 어떤 값보다 큰가?"라는 질문을 데이터로 고릅니다. [실습 예제 13-3]에서 사이킷런이 고를 질문을, 윈도우 8개로 먼저 직접 골라 봅니다. 기준은 질문으로 나눈 두 무리가 얼마나 덜 섞였는지(지니 불순도)입니다.

**❶ 값을 정렬하고 후보 문턱 만들기**

흔들림 8개와 정답(움직임 1, 멈춤 0)을 두고, 이웃한 두 값의 가운데를 후보 문턱으로 삼습니다.

In [ ]:
shake = np.array([0.004, 0.006, 0.005, 0.030, 0.121, 0.090, 0.138, 0.208])
move = np.array([0, 0, 0, 0, 1, 1, 1, 1])      # 1: 움직임, 0: 멈춤
v = np.sort(shake)
cand = (v[:-1] + v[1:]) / 2                    # 이웃한 두 값의 가운데
print("정렬:", v)
print("후보 문턱:", cand.round(4))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.sort(shake)` 흔들림을 작은 것부터 늘어놓습니다.
>
> `(v[:-1] + v[1:]) / 2` 앞 7개와 뒤 7개를 짝지어 더하고 2로 나눕니다. 이웃한 두 값의 가운데 7개가 후보입니다. 그 사이 어디에 문턱을 두어도 나뉘는 결과는 같으므로 가운데만 보면 됩니다.



값이 8개이면 나누는 방법은 7가지뿐입니다. 나무는 이 7가지를 모두 따져 봅니다.

**❷ 문턱마다 지니 불순도와 정확도 계산하기**

후보 문턱마다 왼쪽(≤ 문턱)과 오른쪽(> 문턱)으로 나누고, 두 무리가 얼마나 섞였는지 잽니다.

In [ ]:
import pandas as pd
def gini(y):                                    # 지니 불순도: 섞인 정도
    if len(y) == 0: return 0.0
    p = y.mean()
    return 1 - p**2 - (1 - p)**2
rows = []
for c in cand:
    L, R = move[shake <= c], move[shake > c]
    g = (len(L) * gini(L) + len(R) * gini(R)) / len(move)
    acc = ((shake > c).astype(int) == move).mean()
    rows.append((round(c, 4), len(L), len(R), round(g, 3), acc))
print(pd.DataFrame(rows, columns=["문턱", "왼쪽", "오른쪽", "지니", "정확도"]).to_string(index=False))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `gini(y)` 1 − p² − (1 − p)². p는 무리 안의 움직임 비율입니다. 한 종류만 있으면 0, 반반 섞이면 0.5로 가장 큽니다.
>
> `(len(L) * gini(L) + len(R) * gini(R)) / len(move)` 두 무리의 불순도를 크기로 가중 평균합니다. 작을수록 좋은 질문입니다.
>
> `acc` "문턱보다 크면 움직임"이라고 답할 때 8개 가운데 맞힌 비율입니다.



문턱 0.06에서 지니가 0, 정확도가 1.0입니다. 왼쪽 4개는 모두 멈춤, 오른쪽 4개는 모두 움직임입니다. 끝으로 갈수록 한쪽 무리가 섞여 지니가 커집니다.

**❸ 사이킷런의 나무와 견주기**

같은 8개로 `DecisionTreeClassifier(max_depth=1)`을 학습시키고 고른 질문을 글로 출력합니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
stump = DecisionTreeClassifier(max_depth=1).fit(shake.reshape(-1, 1), move)
print(export_text(stump, feature_names=["흔들림"], decimals=4))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `shake.reshape(-1, 1)` 특징이 하나뿐이어도 사이킷런은 (행, 열) 모양의 표를 받으므로 열 하나짜리로 바꿉니다.
>
> `max_depth=1` 질문을 하나만 하는 나무(스텀프)입니다.
>
> `export_text( )` 나무가 고른 질문과 잎의 답을 글로 보여 줍니다.



사이킷런도 "흔들림 ≤ 0.0600"을 골랐습니다. 손으로 찾은 문턱과 같습니다. 실제 데이터에서는 특징 110개 × 후보 수천 개를 이렇게 모두 따져 가장 좋은 질문을 고르고, 나뉜 무리마다 같은 일을 되풀이해 나무를 키웁니다.

#### [실습 예제 13-3 ★] 특징 110개 만들고 작은 나무로 분류하기

윈도우마다 신호 11개 × 통계 10개 = 특징 110개를 만들고, 질문 하나짜리 나무부터 깊이를 늘려 가며 학습·시험 정확도를 견주시오(03절 3.3).

**❶ 특징 110개 만들기**

In [ ]:
def feats(sig):                            # 신호 11개 × 통계 10개
    chans = [sig[:, :, j] for j in range(9)]
    chans += [np.linalg.norm(sig[:, :, 6:9], axis=2),   # 가속도 크기
              np.linalg.norm(sig[:, :, 3:6], axis=2)]   # 회전 크기
    cols = []
    for c in chans:
        Sp = spec(c)
        band = lambda lo, hi: (Sp[:, (fr >= lo) & (fr < hi)] ** 2).sum(axis=1)
        q75, q25 = np.percentile(c, [75, 25], axis=1)
        cols += [c.mean(1), c.std(1), c.min(1), c.max(1), q75 - q25,
                 np.sqrt((c ** 2).mean(1)), fr[1:][Sp[:, 1:].argmax(1)],
                 band(0.3, 3), band(3, 10), band(10, 25.1)]
    return np.column_stack(cols)

F_tr, F_te = feats(sig_tr), feats(sig_te)
print("특징 표:", F_tr.shape, F_te.shape)
print("윈도우 0, 몸 가속도 x의 평균·표준편차·최솟값·최댓값:",
      F_tr[0, :4].round(3))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `chans` 원신호 9채널에 가속도 크기와 회전(자이로) 크기를 더해 신호 11개를 만듭니다. 크기는 휴대폰 방향과 상관없는 신호입니다.
>
> `for c in chans:` 신호 하나 `c`는 (윈도우, 128) 표입니다. 아래 통계는 모두 `axis=1`(시간 방향)로 계산해 윈도우마다 숫자 하나를 냅니다.
- 통계 10개: 평균(자세), 표준편차(흔들림), 최솟값·최댓값, 사분위 범위(`q75 - q25`, 가운데 절반이 퍼진 폭), RMS(`np.sqrt((c ** 2).mean(1))`, 제곱 평균의 제곱근, 전체 세기), 가장 센 주파수(리듬), 세 주파수 띠의 에너지(느린 0.3~3Hz, 중간 3~10Hz, 빠른 10~25Hz).
> **[코드 읽기] 한 줄씩**
>
> `band = lambda lo, hi: …` lo~hi Hz 칸들의 세기를 제곱해 더하는 작은 함수입니다(아래 [코드 읽기]).
>
> `np.column_stack(cols)` 숫자 줄(윈도우 수 길이) 110개를 열로 나란히 세워 (윈도우, 110) 표를 만듭니다. 열 순서는 "신호 1의 통계 10개, 신호 2의 통계 10개, …"입니다.



윈도우 하나(128칸 × 9채널 = 숫자 1,152개)가 숫자 110개로 줄었습니다. 첫 윈도우는 서 있는 중이라 몸 가속도가 거의 0 근처에서 아주 조금만 흔들립니다. 특징은 사람이 이름을 붙여 읽을 수 있습니다. "10번째 신호의 표준편차"처럼 모델이 무엇을 보았는지 말할 수 있다는 것이 원신호를 그대로 넣는 방법보다 나은 점입니다.

> **[코드 읽기] `lambda`와 `np.percentile( )`**
>
> `band = lambda lo, hi: …`는 이름 없는 작은 함수를 한 줄로 만듭니다. `def band(lo, hi): return …`와 같습니다. `band(0.3, 3)`이라고 부르면 0.3~3Hz 칸들의 세기를 제곱해 더한 값을 돌려줍니다. `(fr >= lo) & (fr < hi)`는 그 범위의 칸만 참인 배열이라, 대괄호에 넣으면 그 칸들만 고릅니다. `np.percentile(c, [75, 25], axis=1)`은 윈도우마다 75%와 25% 위치의 값을 한 번에 돌려줍니다. 둘의 차이(사분위 범위)는 4장의 상자그림에서 상자의 길이입니다.

**❷ 질문 하나짜리 나무: 움직이나, 멈췄나**

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
j = 9 * 10 + 1                             # 10번째 신호(가속도 크기)의 표준편차
move_tr, move_te = y_tr <= 2, y_te <= 2    # 0~2번 동작 = 움직이는 중
stump = DecisionTreeClassifier(max_depth=1)
stump.fit(F_tr[:, [j]], move_tr)
print(export_text(stump, feature_names=["흔들림"], decimals=4))
print("움직임/멈춤 정확도:", stump.score(F_te[:, [j]], move_te))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `j = 9 * 10 + 1` 신호 9개 다음, 10번째 신호(가속도 크기)의 통계 10개 가운데 두 번째(표준편차) 열 번호, 곧 91번입니다.
>
> `y_tr <= 2` 걷기·계단 오르기·계단 내려가기(0~2)면 참(움직임), 나머지면 거짓(멈춤)인 두 가지 정답을 새로 만듭니다.
>
> `max_depth=1` 질문을 한 번만 하는 나무(그루터기, stump)입니다.
>
> `F_tr[:, [j]]` 대괄호를 두 겹 써서 (윈도우, 1) 표 모양을 유지합니다. 사이킷런은 특징을 늘 2차원 표로 받습니다.
>
> `export_text( )` 나무가 고른 질문과 답을 글자로 보여 줍니다(아래 [잠깐만요] 상자). `.score( )`는 정확도입니다.



나무는 "흔들림이 0.0822보다 큰가?"를 골랐고, 처음 보는 9명의 윈도우 2,947개를 하나도 틀리지 않았습니다(1.0 = 100%). 13-2 ❷의 흔들림 표에서 움직임(0.12 이상)과 멈춤(0.006 이하) 사이에 문턱이 놓인 것입니다. 가장 단순한 모델이 이미 절반의 문제를 풀었습니다. 남은 어려움은 걷기와 계단, 앉기와 서기를 가르는 일입니다.

> **[잠깐만요] 결정나무 읽는 법**
>
> 결정나무는 "특징 ≤ 문턱이면 왼쪽, 아니면 오른쪽"이라는 질문을 이어 붙인 것입니다. `export_text( )`의 한 줄 들여쓰기가 질문 한 번이고, 맨 끝의 `class`가 그 길로 내려온 데이터에 내리는 답입니다. 나무는 질문마다 "두 쪽이 가장 깔끔하게 갈리는" 특징과 문턱을 데이터에서 스스로 고릅니다(10장).
>
> 은행의 대출 심사로 치면 "연 소득 ≤ 3,000만 원?" → "연체 기록 있음?" → 거절처럼 읽힙니다. 그래서 나무는 결과를 사람에게 설명해야 하는 일에 많이 씁니다. 대신 질문이 깊어질수록 읽기 어려워지고, 학습 데이터의 우연까지 외우기 쉽습니다(③과 [잠깐만요] 과대적합).

**❸ 질문을 늘리면: 깊이별 학습·시험 정확도**

In [ ]:
rows = []
for d in [2, 3, 4, 6, 8, 12, None]:        # None = 끝까지 키우기
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(F_tr, y_tr)
    rows.append([d or "끝까지", t.tree_.node_count,
                 t.score(F_tr, y_tr), t.score(F_te, y_te)])
depth = pd.DataFrame(rows, columns=["깊이", "마디", "학습", "시험"])
print(depth.round(3).to_string(index=False))

plt.plot(depth["깊이"].astype(str), depth["학습"], marker="o",
         color="#9a9a9a", label="학습")
plt.plot(depth["깊이"].astype(str), depth["시험"], marker="o",
         color="#C8234B", label="처음 보는 9명")
plt.xlabel("나무의 깊이 (질문을 몇 번 이어 하나)")
plt.ylabel("6동작 정확도")
plt.legend()
plt.gcf().set_size_inches(6, 3)
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `for d in […, None]` 깊이를 바꿔 가며 같은 특징 110개로 여섯 동작 나무를 만듭니다. `None`은 "더 나눌 수 없을 때까지"입니다.
>
> `random_state=0` 점수가 같은 질문이 여럿일 때 고르는 순서를 고정해, 다시 돌려도 같은 나무가 나옵니다([잠깐만요] random_state).
>
> `t.tree_.node_count` 나무의 마디(질문 + 답) 수, 곧 모델의 크기입니다.
>
> `d or "끝까지"` `d`가 `None`이면 뒤의 글자를 씁니다. 표를 읽기 좋게 하려는 것입니다.
>
> `t.score(F_tr, y_tr)`와 `t.score(F_te, y_te)` 같은 나무를 배운 데이터와 처음 보는 데이터로 각각 채점합니다. 둘을 나란히 보아야 과대적합이 보입니다.



*(그래프: 나무의 깊이에 따른 정확도. 학습(회색)은 깊어질수록 100%로 오르지만, 처음 보는 9명(빨강)은 깊이 8의 86.7%에서 멈추고 조금 내려간다.)*

깊이 2에서 4까지는 학습과 시험이 함께 오릅니다. 깊이 8부터 학습은 96%→100%로 계속 오르는데 시험은 86.7%에서 84.4%로 오히려 내려갑니다. 학습 데이터의 우연까지 외우기 시작한 것입니다. 이 표에서 고른다면 깊이 6~8, 마디 57~141개입니다. 다만 깊이를 시험 점수로 고르면 시험 문제를 보고 답을 고친 셈이 되므로, 제대로 하려면 검증 데이터로 고릅니다(코랩 실습 13-9).

> **[잠깐만요] 나무가 깊어지면: 과대적합**
>
> 나무를 깊게 키울수록 학습 데이터는 점점 더 잘 맞힙니다. 같은 특징으로 재 보면 깊이 4는 학습 88.0%·시험 83.1%, 깊이 12는 학습 99.5%·시험 84.3%, 끝까지 키우면 학습 100%·시험 84.4%입니다. 학습 점수만 오르고 시험 점수는 멈춘 상태, 곧 문제집의 답을 외웠지만 새 문제는 못 푸는 상태를 **과대적합**(overfitting)이라고 합니다.
>
> 그래서 모델의 크기(깊이, 마디 수)는 학습 점수가 아니라 처음 보는 데이터의 점수로 고릅니다. 학습과 시험 점수의 차이가 벌어지기 시작하는 곳이 멈출 자리입니다. 집값 예측의 회귀 나무든 스팸 분류든 같습니다.

#### [알고리즘 따라 하기 13-3] 나무 세 그루의 다수결 직접 해 보기

랜덤 포레스트는 데이터를 조금씩 다르게 뽑아 나무를 여러 그루 키우고, 다수결로 답합니다. [실습 예제 13-4]의 200그루 숲을, 윈도우 8개와 나무 3그루로 먼저 따라 해 봅니다.

**❶ 나무마다 표본 뽑기(복원 추출)**

윈도우 8개(특징: 흔들림, x축 평균)에서 나무마다 8개를 다시 뽑습니다. 뽑은 것을 되돌려 놓고 뽑으므로 같은 윈도우가 여러 번 나오기도 하고, 빠지기도 합니다.

In [ ]:
X = np.array([[0.121, 1.00], [0.138, 0.98], [0.208, 0.97], [0.090, 0.99],
              [0.005, 0.99], [0.006, 0.96], [0.004, 0.07], [0.005, 0.10]])
y = np.array([1, 1, 1, 1, 0, 0, 0, 0])          # 1: 움직임, 0: 멈춤
rng = np.random.default_rng(1)
bags = [rng.integers(0, len(X), len(X)) for _ in range(3)]   # 복원 추출
for b, idx in enumerate(bags):
    print(f"나무 {b + 1}의 표본 번호: {sorted(idx.tolist())}")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `rng.integers(0, len(X), len(X))` 0~7 가운데에서 8번 뽑습니다. 같은 번호가 또 나올 수 있습니다(복원 추출, 부트스트랩).
>
> `default_rng(1)` 난수의 씨앗을 고정해 다시 실행해도 같은 표본이 나오게 합니다.



나무 2의 표본에는 4번, 5번(서 있는 멈춤 윈도우)이 하나도 없습니다. 이 나무는 "서 있는 멈춤"을 본 적이 없습니다.

**❷ 나무마다 학습하고 새 윈도우 판단하기**

나무마다 특징 하나만 보고(1·3번 나무는 흔들림, 2번 나무는 x축 평균) 질문 하나짜리 나무를 키운 뒤, 새 윈도우 3개(걷기? 서기? 눕기?)를 판단합니다.

In [ ]:
new = np.array([[0.150, 0.95], [0.006, 0.97], [0.004, 0.08]])   # 걷기? 서기? 눕기?
trees, votes = [], []
for b, idx in enumerate(bags):
    col = b % 2                                 # 나무마다 다른 특징 하나만 본다
    tr = DecisionTreeClassifier(max_depth=1).fit(X[idx][:, [col]], y[idx])
    trees.append((tr, col))
    pred = tr.predict(new[:, [col]])
    votes.append(pred)
    print(f"나무 {b + 1}(특징 {['흔들림', 'x 평균'][col]}): 문턱 {tr.tree_.threshold[0]:.3f}, 새 윈도우 판단 {pred}")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `col = b % 2` 나무마다 보는 특징을 바꿉니다. 실제 랜덤 포레스트도 질문할 때마다 특징을 무작위로 일부만 봅니다.
>
> `X[idx][:, [col]]` 그 나무가 뽑은 표본에서 고른 특징 열만 꺼냅니다.
>
> `tr.tree_.threshold[0]` 그 나무가 고른 문턱입니다. 표본이 다르니 나무마다 문턱이 조금씩 다릅니다.



나무 2는 두 번째 윈도우(서기)를 움직임(1)으로 틀렸습니다. 표본에 서 있는 멈춤이 없어 "x축 평균이 크면 움직임"이라고 배웠기 때문입니다.

**❸ 다수결로 답하기**

세 나무의 답을 모아 2표 이상이면 움직임으로 답합니다.

In [ ]:
votes = np.array(votes)
print("찬성(움직임) 표 수:", votes.sum(axis=0))
print("다수결:", (votes.sum(axis=0) >= 2).astype(int))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `votes.sum(axis=0)` 윈도우마다 "움직임" 표를 셉니다.
>
> `>= 2` 세 그루 가운데 둘 이상이면 다수결로 움직임입니다.



나무 2가 틀린 서기 윈도우도 나머지 두 그루가 바로잡아 다수결은 [1 0 0]으로 모두 맞혔습니다. 한 그루는 약해도, 서로 다른 데이터로 배운 여러 그루가 모이면 실수가 상쇄됩니다. 이것이 랜덤 포레스트입니다.

#### [실습 예제 13-4 ★] 처음 보는 사람으로 시험하기

랜덤 포레스트로 (1) 사람 단위로 나눈 시험과 (2) 윈도우를 무작위로 나눈 시험의 정확도를 견주고, 사람별 정확도와 혼동행렬로 결과 뒤를 들여다보시오(03절 3.4).

**❶ 사람 단위 시험: 학습 21명으로 배우고 처음 보는 9명으로 채점**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(F_tr, y_tr)                         # 학습 21명으로 배우고
p_rf = rf.predict(F_te)                    # 처음 보는 9명의 답을 낸다
print("예측 앞 10개:", p_rf[:10], " 정답:", y_te[:10])
print("처음 보는 9명으로 시험:", round((p_rf == y_te).mean() * 100, 2), "%")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `RandomForestClassifier(…)` 나무 여러 그루의 다수결 모델입니다(아래 [잠깐만요] 상자). `n_estimators=200`은 나무 200그루, `random_state=42`는 무작위 뽑기의 씨앗, `n_jobs=-1`은 컴퓨터의 모든 코어를 쓰라는 뜻입니다.
>
> `rf.fit(F_tr, y_tr)` 특징 표와 정답으로 배웁니다. 연구팀이 나눈 그대로라 학습에는 21명만 있습니다.
>
> `rf.predict(F_te)` 시험 윈도우 2,947개마다 0~5 가운데 하나를 답합니다.
>
> `(p_rf == y_te).mean( )` 맞으면 참(1), 틀리면 거짓(0)이니 평균이 곧 정확도입니다.



앞 10개는 모두 4번(서기)으로 맞혔습니다. 처음 보는 9명 전체로는 90.23%입니다. 질문 8번짜리 나무 하나(86.7%)보다 3.5%p 높습니다. 이 숫자가 "새 사용자에게 쓸 때"의 점수입니다.

> **[잠깐만요] 사이킷런 모델은 쓰는 법이 모두 같다**
>
> 사이킷런의 모델은 종류가 달라도 세 단계로 씁니다. ① 만들기: `모델 = 모델이름(설정)`, ② 배우기: `모델.fit(X학습, y학습)`, ③ 쓰기: `모델.predict(X새것)` 또는 채점까지 한 번에 `모델.score(X시험, y시험)`. 이 장에서 쓴 결정나무, 랜덤 포레스트, 로지스틱 회귀(코랩 실습 13-8), 신경망(코랩 실습 13-11)이 모두 같습니다. 8장의 선형 회귀도 같았습니다.
>
> 그래서 모델을 바꿔 보는 데는 첫 줄 하나만 고치면 됩니다. 확률이 필요하면 `predict_proba( )`(13-6), 숫자의 눈금을 맞추는 전처리는 `fit_transform( )`(코랩 실습 13-10)을 씁니다. `X`는 늘 2차원 표(행 = 사례, 열 = 특징), `y`는 1차원 정답입니다.

> **[잠깐만요] 랜덤 포레스트: 나무 여러 그루의 다수결**
>
> 나무 한 그루는 학습 데이터가 조금만 바뀌어도 모양이 크게 달라집니다. **랜덤 포레스트**(random forest)는 나무마다 (1) 학습 데이터를 중복을 허락해 무작위로 다시 뽑고, (2) 질문할 때마다 특징의 일부만 보게 해서 서로 조금씩 다른 나무를 많이 만든 뒤, 그 답을 다수결로 모읍니다(10장).
>
> 한 사람의 판단보다 여러 사람의 투표가 덜 흔들리는 것과 같습니다. 특징이 많은 표 데이터에서 따로 손보지 않아도 잘 맞아서, 고객 이탈, 대출 심사, 센서 분류 어디서나 먼저 세워 보는 기준 모델로 많이 씁니다. 대신 나무를 모두 저장해야 해서 모델이 큽니다(코랩 실습 13-11).

**❷ 무작위로 섞어 나누면(누수가 있는 시험)**

모델과 특징은 그대로 두고, 나누는 방법만 "윈도우를 무작위로"로 바꿉니다.

In [ ]:
from sklearn.model_selection import train_test_split
F_all = np.r_[F_tr, F_te]
A, B, ya, yb = train_test_split(F_all, y_all,
                                test_size=len(y_te) / len(y_all),
                                random_state=42, stratify=y_all)
rf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf2.fit(A, ya)
print("시험 윈도우 수:", len(yb))
print("윈도우를 무작위로 나눠 시험:", round(rf2.score(B, yb) * 100, 2), "%")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.r_[F_tr, F_te]` 학습과 시험을 다시 하나로 합칩니다. 사람 정보는 쓰지 않습니다.
>
> `train_test_split(X, y, …)` 줄을 무작위로 섞어 둘로 나눕니다. 돌려주는 순서는 (X학습, X시험, y학습, y시험)입니다.
>
> `test_size=len(y_te) / len(y_all)` 시험 비율을 원래와 같게(2,947 ÷ 10,299 ≈ 28.6%) 맞춥니다.
>
> `stratify=y_all` 두 묶음의 동작 비율을 같게 맞춥니다(아래 [잠깐만요] 상자).
>
> `rf2.score(B, yb)` `predict` 뒤 정답과 견주는 일을 한 번에 합니다.



시험 수(2,947개), 동작 비율, 모델, 특징이 모두 ①과 같고 나누는 방법만 다른데 7.4%p가 높습니다. 한 번에 하나만 바꾸었으니 차이의 원인은 나누는 방법입니다. 무작위로 나누면 시험 윈도우 대부분의 바로 옆 윈도우가 학습에 있고, 둘은 숫자 64개를 함께 갖습니다(1.3). 같은 사람의 걸음걸이도 이미 배운 상태입니다. 97.66%는 "이미 본 사람의 다음 2.56초"를 맞히는 점수입니다.

> **[잠깐만요] 층화 추출(stratify): 비율을 지키며 나누기**
>
> 반 학생 100명 가운데 안경 쓴 학생이 10명이라고 합시다. 무작위로 20명을 뽑으면 우연히 안경 쓴 학생이 0명일 수도 있습니다. **층화 추출**은 무리(층)마다 비율을 지키며 뽑는 방법입니다. 20명을 뽑으면 안경 쓴 학생 2명, 아닌 학생 18명이 들어갑니다.
>
> 고장(3.39%), 스팸, 카드 부정 사용처럼 드문 범주가 있는 데이터는 무작위로 나누면 시험 묶음에 그 범주가 너무 적거나 많아질 수 있습니다. 그래서 `train_test_split(…, stratify=y)`로 나눕니다. 다만 층화 추출은 비율만 맞출 뿐, 같은 사람의 기록이 섞이는 누수는 막지 못합니다. 그것은 그룹 단위 나누기의 몫입니다.

**❸ 사람별 정확도: 평균 뒤의 가장 나쁜 사람**

In [ ]:
acc = pd.Series({s: (p_rf[s_te == s] == y_te[s_te == s]).mean() * 100
                 for s in sorted(set(s_te))})
print("사람별 정확도:", acc.round(1).to_dict())
print("가장 나쁜 사람:", acc.idxmin(), "번", round(acc.min(), 1), "%")

acc.sort_values().plot.bar(color="#9a9a9a", figsize=(6, 3), rot=0)
plt.axhline((p_rf == y_te).mean() * 100, color="#C8234B", linestyle="--")
plt.xlabel("시험용 사람 번호 (점선: 9명 평균)")
plt.ylabel("정확도 (%)")
plt.ylim(60, 100)
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `s_te == s` 시험 윈도우 가운데 사람 s의 것만 참입니다. 예측과 정답에 같은 조건을 걸어 그 사람의 윈도우만 견줍니다.
>
> `{s: … for s in …}` 사전 만들기 문법으로 9명의 정확도를 한 줄에 모읍니다. `pd.Series( )`로 감싸면 번호가 붙은 숫자 줄이 됩니다.
>
> `acc.idxmin( )` 가장 작은 값의 번호(사람)를 돌려줍니다.
>
> `sort_values( )` 낮은 사람부터 막대를 세웁니다. `plt.axhline( )`은 9명 평균에 가로 점선을 긋습니다. `plt.ylim(60, 100)`은 세로축을 60~100으로 잘라 차이를 잘 보이게 합니다.



*(그래프: 시험 9명의 사람별 정확도. 점선은 9명 평균 90.23%다. 10번 사람은 77.9%, 24번 사람은 99.5%였다.)*

평균은 90.23%지만 10번 사람에게는 열에 두 번 넘게 틀립니다. 24번 사람과는 21.6%p 차이입니다. 사람마다 걸음걸이와 휴대폰을 찬 위치가 다르기 때문입니다. 9명뿐이라 "가장 나쁜 사람"도 운에 따라 달라질 수 있습니다. 그래서 제품을 낼 때는 평균과 함께 가장 나쁜 사용자, 하위 10%의 정확도를 따로 적습니다.

**❹ 혼동행렬: 무엇을 무엇으로 착각했나**

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_te, p_rf)          # 행: 정답, 열: 예측
print(pd.DataFrame(cm, index=ACT, columns=ACT).to_string())
print("동작별 재현율(%):", (cm.diagonal() / cm.sum(axis=1) * 100).round(1))

plt.imshow(cm, cmap="Reds")                # 숫자가 클수록 진한 칸
for a in range(6):
    for b in range(6):
        plt.text(b, a, cm[a, b], ha="center", va="center", fontsize=9)
plt.xticks(range(6), ["걷기", "오르기", "내려가기", "앉기", "서기", "눕기"],
           rotation=45)
plt.yticks(range(6), ["걷기", "오르기", "내려가기", "앉기", "서기", "눕기"])
plt.xlabel("모델의 답 (예측)")
plt.ylabel("실제 동작 (정답)")
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `confusion_matrix(정답, 예측)` 6 × 6 표를 돌려줍니다. `cm[3, 4]`는 정답이 3번(앉기)인데 4번(서기)으로 답한 개수입니다. 인수 순서(정답이 먼저)를 바꾸면 행과 열이 뒤집히니 조심합니다.
>
> `pd.DataFrame(cm, index=ACT, columns=ACT)` 행과 열에 동작 이름을 붙여 읽기 쉽게 합니다.
>
> `cm.diagonal( ) / cm.sum(axis=1)` 대각선(맞힌 수)을 행의 합(그 동작의 실제 개수)으로 나누면 동작별 재현율입니다.
- `plt.imshow( )`와 이중 `for` 문: 표를 색칠하고, 칸마다 `plt.text(열, 행, 숫자)`로 숫자를 적습니다. 그림 좌표는 (가로, 세로) 순이라 `(b, a)`로 씁니다.



*(그래프: 처음 보는 9명의 혼동행렬. 대각선이 맞힌 것이다. 눕기는 537개 모두 맞혔고, 앉기와 서기(88개, 47개), 계단 내려가기와 계단 오르기(57개)를 많이 헷갈렸다.)*

틀린 곳이 고르게 퍼져 있지 않습니다. 오른쪽 아래 멈춘 동작 묶음과 왼쪽 위 움직이는 동작 묶음 사이의 칸은 모두 0입니다. ❷의 질문 하나가 가른 두 무리를 모델도 섞지 않았습니다. 틀림은 자세가 비슷한 앉기·서기(135개)와 흔들림이 비슷한 계단 오르기·내려가기(61개)에 몰렸습니다. 계단 내려가기는 79.8%만 찾았습니다. 다음에 고칠 곳이 바로 여기입니다(코랩 실습 13-7).

> **[잠깐만요] 혼동행렬 읽기**
>
> **혼동행렬**(confusion matrix)은 "정답이 이것일 때 모델은 무엇이라고 답했나"를 센 표입니다. 행은 정답, 열은 모델의 답이고, 대각선이 맞힌 개수입니다. 대각선 밖의 큰 숫자가 모델이 헷갈리는 짝입니다.
>
> 한 행을 그 행의 합으로 나누면 그 동작의 재현율, 곧 "실제 그 동작 가운데 몇 %를 찾았나"입니다. 계단 내려가기는 420개 가운데 335개, 79.8%만 찾았습니다. 전체 정확도 90.23% 하나로는 보이지 않는 약점입니다.
>
> 둘로 나누는 문제(스팸/정상, 병/정상)에서는 2 × 2 표가 되고, 3.7의 정밀도·재현율·F1이 모두 이 표의 네 칸으로 계산됩니다. 분류 모델을 만들면 정확도 다음에 꼭 이 표를 찍어 보세요.

#### [코랩 실습 13-5] 사람 수를 늘리면

> 📒 **코랩 실습**: 책에는 문제와 결과 한 줄만 있고, 코드·설명·결과는 여기에 있습니다.

학습에 쓰는 사람을 1, 3, 6, 12, 21명으로 바꾸며 처음 보는 9명의 정확도를 구하시오. 그다음 윈도우 수를 800개로 맞추고 사람 수만 바꾸어 견주시오(03절 3.5).

**❶ 사람 수를 바꾸며 다섯 번씩 뽑기**

In [ ]:
rng = np.random.default_rng(0)             # 씨앗 0: 매번 같은 사람이 뽑힌다
people = np.array(sorted(set(s_tr)))
RF = lambda: RandomForestClassifier(n_estimators=200, random_state=42,
                                    n_jobs=-1)
res = {}
for k in [1, 3, 6, 12, 21]:                # 학습에 쓰는 사람 수
    accs = []
    for rep in range(5 if k < 21 else 1):  # 21명은 한 가지뿐
        m = np.isin(s_tr, rng.choice(people, k, replace=False))
        accs.append((RF().fit(F_tr[m], y_tr[m]).predict(F_te)
                     == y_te).mean() * 100)
    res[k] = accs
    print(f"{k:2d}명: 평균 {np.mean(accs):.1f}%"
          f" ({min(accs):.1f}~{max(accs):.1f}%)")

for k, a in res.items():
    plt.scatter([k] * len(a), a, color="#9a9a9a", s=22)
plt.plot(list(res), [np.mean(a) for a in res.values()], marker="o",
         color="#C8234B", label="다섯 번의 평균")
plt.xlabel("학습에 쓴 사람 수 (회색 점: 뽑을 때마다의 정확도)")
plt.ylabel("처음 보는 9명 정확도 (%)")
plt.xticks([1, 3, 6, 12, 21])
plt.legend(loc="lower right")
plt.gcf().set_size_inches(5.5, 3)
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.random.default_rng(0)` 넘파이의 난수 생성기입니다. 씨앗 0에서 출발하므로 몇 번을 돌려도 같은 사람이 뽑힙니다.
>
> `RF = lambda: …` 같은 설정의 랜덤 포레스트를 새로 만드는 작은 함수입니다. `RF( )`라고 부를 때마다 새 모델이 나옵니다.
>
> `rng.choice(people, k, replace=False)` 학습용 21명 가운데 k명을 겹치지 않게 뽑습니다.
>
> `np.isin(s_tr, 뽑은 사람)` 학습 윈도우마다 그 주인이 뽑힌 사람인지 참·거짓으로 표시합니다. `F_tr[m]`은 뽑힌 사람의 윈도우만 남깁니다.
>
> `range(5 if k < 21 else 1)` 누구를 뽑느냐는 운이므로 다섯 번 뽑아 평균과 범위를 봅니다. 21명은 모두를 쓰는 한 가지뿐입니다.
>
> `plt.scatter([k] * len(a), a)` 같은 가로 위치 k에 다섯 번의 점수를 점으로 찍어 퍼짐을 보여 줍니다.



*(그래프: 학습에 쓴 사람 수마다 다섯 번 뽑은 정확도(회색 점)와 그 평균(빨강). 1명이면 71.0~82.6%로 누구를 뽑았느냐에 따라 크게 흔들리고, 사람이 늘수록 높아지며 덜 흔들린다.)*

사람이 늘수록 평균이 오르고(75.1% → 90.2%) 범위는 좁아집니다(11.6%p → 2.6%p). 1명만 쓰면 누구를 뽑았느냐에 따라 11%p 넘게 달라집니다. 21명에서도 곡선이 아직 오르고 있으니 사람을 더 모으면 나아질 것으로 보입니다([잠깐만요] 학습 곡선). 그런데 사람이 늘면 윈도우 수도 함께 늘었습니다. 무엇 덕분일까요?

**❷ 같은 800개, 사람 수만 다르게**

In [ ]:
for k in [3, 21]:                          # 윈도우 800개를 3명 또는 21명에게서
    accs = []
    for rep in range(5):
        pick = rng.choice(people, k, replace=False)
        idx = np.where(np.isin(s_tr, pick))[0]
        idx = rng.choice(idx, 800, replace=False)
        accs.append((RF().fit(F_tr[idx], y_tr[idx]).predict(F_te)
                     == y_te).mean() * 100)
    print(f"윈도우 800개, {k}명에게서: {np.mean(accs):.1f}%")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `np.where(참거짓)[0]` 참인 자리의 번호들을 돌려줍니다. 뽑힌 사람의 윈도우 번호 목록입니다.
>
> `rng.choice(idx, 800, replace=False)` 그 가운데 800개만 무작위로 남깁니다. 3명이든 21명이든 학습 데이터의 양이 같아집니다.
- 나머지는 ①과 같습니다. 같은 `rng`를 이어 쓰므로 ①을 먼저 실행해야 책과 같은 숫자가 나옵니다.



양이 같아도 21명에게서 모은 쪽이 6.8%p 높습니다. 한 사람에게서 더 오래 기록하면 그 사람의 버릇을 더 깊이 배울 뿐입니다. 데이터의 양보다 다양함이 먼저입니다. 이렇게 "다른 조건은 모두 같게 두고 하나만 바꾸는" 비교를 해야 차이의 원인을 말할 수 있습니다.

> **[잠깐만요] random_state: 같은 결과를 다시 얻기**
>
> 컴퓨터의 "무작위"는 씨앗(seed)이라는 숫자에서 시작하는 정해진 수열입니다. 씨앗이 같으면 몇 번을 돌려도 같은 사람이 뽑히고 같은 나무가 자랍니다. `np.random.default_rng(0)`의 0, `random_state=42`의 42가 씨앗입니다. 숫자 자체에는 뜻이 없습니다.
>
> 씨앗을 정해 두면 책의 숫자를 독자가 그대로 다시 얻을 수 있고, 모델을 고친 효과와 우연을 가를 수 있습니다. 다만 씨앗 하나의 결과는 운일 수 있으므로, 중요한 비교는 이 실습처럼 여러 번 반복해 평균과 범위를 함께 보고합니다.

> **[잠깐만요] 학습 곡선: 데이터를 더 모을 가치가 있을까**
>
> 가로축에 학습 데이터의 양(여기서는 사람 수), 세로축에 처음 보는 데이터의 점수를 그린 그래프를 **학습 곡선**(learning curve)이라고 합니다. 곡선이 아직 오르는 중이면 데이터를 더 모을 가치가 있고, 평평해졌으면 데이터보다 특징이나 모델을 바꾸는 편이 낫습니다.
>
> 이 그래프는 21명에서도 아직 오르고 있으니 사람을 더 모으면 나아질 것으로 보입니다. 설문 응답, 고객 기록, 병원 영상처럼 데이터를 모으는 데 돈과 시간이 드는 일이라면, 더 모으기 전에 먼저 학습 곡선부터 그려 봅니다.

#### [알고리즘 따라 하기 13-4] 정밀도·재현율·F1을 손으로 세기

[실습 예제 13-6]은 드문 고장을 찾으므로 정확도 대신 정밀도, 재현율, F1로 견줍니다. 작업 10건(고장 3건)과 모델이 낸 고장 확률로 먼저 직접 세어 봅니다.

**❶ 문턱 0.5로 예측하고 TP·FP·FN 세기**

고장 확률이 0.5 이상이면 "고장"으로 예측하고, 맞힌 경보(TP), 헛경보(FP), 놓친 고장(FN)을 셉니다.

In [ ]:
y_true = np.array([1, 0, 0, 1, 0, 0, 1, 0, 0, 0])   # 1: 고장
score = np.array([0.9, 0.2, 0.6, 0.7, 0.1, 0.3, 0.4, 0.05, 0.55, 0.15])
pred = (score >= 0.5).astype(int)
TP = ((pred == 1) & (y_true == 1)).sum()
FP = ((pred == 1) & (y_true == 0)).sum()
FN = ((pred == 0) & (y_true == 1)).sum()
print("예측:", pred, " TP:", TP, "FP:", FP, "FN:", FN)

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `(score >= 0.5).astype(int)` 참·거짓을 1·0으로 바꾼 예측입니다.
>
> `(pred == 1) & (y_true == 1)` 예측도 고장, 실제도 고장인 것(TP)입니다. `&`는 "그리고"입니다.
>
> `FP`, `FN` FP는 예측만 고장(헛경보), FN은 실제만 고장(놓침)입니다.



경보 4건 가운데 맞힌 것은 2건(TP), 헛경보 2건(FP)이고, 확률 0.4였던 고장 1건을 놓쳤습니다(FN).

**❷ 정밀도·재현율·F1 계산하고 사이킷런과 견주기**

세어 둔 숫자로 세 지표를 계산하고, 사이킷런 함수의 답과 맞춰 봅니다.

In [ ]:
prec, rec = TP / (TP + FP), TP / (TP + FN)
f1 = 2 * prec * rec / (prec + rec)
print(f"정밀도 {prec:.3f}, 재현율 {rec:.3f}, F1 {f1:.3f}")
from sklearn.metrics import precision_score, recall_score, f1_score
print("사이킷런:", round(precision_score(y_true, pred), 3), round(recall_score(y_true, pred), 3), round(f1_score(y_true, pred), 3))

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `TP / (TP + FP)` 정밀도: 울린 경보 가운데 진짜 고장의 비율. "울리면 믿을 만한가?"
>
> `TP / (TP + FN)` 재현율: 실제 고장 가운데 찾은 비율. "고장을 빠뜨리지 않는가?"
>
> `2 * prec * rec / (prec + rec)` F1: 둘의 조화 평균. 하나라도 낮으면 같이 낮아집니다.



정밀도 0.5, 재현율 0.667, F1 0.571로 사이킷런과 같습니다.

**❸ 문턱을 바꾸면**

문턱을 0.3, 0.5, 0.7로 바꾸며 경보 수와 두 지표를 봅니다.

In [ ]:
for th in [0.3, 0.5, 0.7]:
    p = (score >= th).astype(int)
    print(f"문턱 {th}: 경보 {p.sum()}건, 정밀도 {precision_score(y_true, p):.2f}, 재현율 {recall_score(y_true, p):.2f}")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `for th in [0.3, 0.5, 0.7]:` 같은 확률에 문턱만 바꿉니다. 모델은 그대로입니다.



문턱을 낮추면 경보가 늘어 재현율이 1.0이 되지만 헛경보도 늘고, 높이면 정밀도가 1.0이 되지만 고장을 놓칩니다. 어느 쪽이 나은지는 놓친 고장과 헛경보의 손해로 정합니다([잠깐만요] 확률을 내는 모델과 문턱값).

#### [실습 예제 13-6 ★] 공장 데이터: 너무 깨끗한지 보고 물리 특징 더하기

AI4I 데이터를 불러와 (1) 단위를 아는 특징 세 개를 만들고, (2) 설명서의 고장 규칙과 라벨을 맞대 보고, (3) 센서값 그대로와 물리 특징을 더한 모델을 정밀도·재현율·F1로 견주고, (4) 출력 구간별 고장률을 그리시오(03절 3.6~3.7).

**❶ 불러오고 첫 세 줄 보기**

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/lingchenlijing/"
                 "AI4I-2020-Predictive-Maintenance-Dataset/master/"
                 "data/ai4i2020.csv")
df.columns = [c.replace("﻿", "") for c in df.columns]
print(df.shape, " 고장:", df["Machine failure"].sum(), "건")
print(df.iloc[:3, 2:9].to_string())

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `pd.read_csv(인터넷 주소)` 판다스는 파일 경로 대신 주소를 주어도 바로 읽습니다. 긴 주소는 괄호 안에서 글자를 이어 써서 여러 줄로 나눴습니다.
>
> `c.replace("﻿", "")` 일부 CSV 파일은 맨 앞에 눈에 안 보이는 표시(BOM)가 붙어 첫 열 이름이 달라집니다. 열 이름에서 지워 둡니다.
>
> `df["Machine failure"].sum( )` 고장이면 1, 아니면 0인 열이라 더하면 고장 건수입니다.
>
> `df.iloc[:3, 2:9]` 번호로 고르기입니다. 앞 3줄, 2번부터 8번 열(종류, 온도 둘, 회전수, 토크, 마모, 고장)만 봅니다.



작업 10,000건, 열 14개, 고장 339건(3.39%)입니다. 열 이름에 단위([K], [rpm], [Nm], [min])가 적혀 있습니다. 공기 온도 298K는 약 25℃입니다. 단위가 이름에 붙어 있는 데이터는 친절한 데이터입니다. 그렇지 않으면 설명서부터 찾아 열마다 단위를 적어 두십시오(3.7의 화성 탐사선).

**❷ 단위를 아는 특징 세 개 만들기**

In [ ]:
df["dT"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = (df["Torque [Nm]"] * df["Rotational speed [rpm]"]
               * 2 * np.pi / 60)           # (식 13-5) W
df["strain"] = df["Tool wear [min]"] * df["Torque [Nm]"]
print(df[["dT", "power", "strain"]].describe().loc[["mean", "min", "max"]]
      .round(1).to_string())

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `df["dT"] = …` 새 열을 만듭니다. 열끼리의 뺄셈·곱셈은 10,000줄에 한 번에 적용됩니다.
- `dT`(K): 공정 온도 - 공기 온도. 열이 얼마나 잘 빠지는지입니다. 온도 차가 작으면 열이 쌓입니다.
- `power`(W): 토크 × 회전수 × 2π ÷ 60. (식 13-5)로 rpm을 1초당 각도(rad/s)로 바꿔 곱한 기계의 출력입니다.
- `strain`(min·Nm): 공구 마모 시간 × 토크. 닳은 공구가 받는 부담입니다.
> **[코드 읽기] 한 줄씩**
>
> `describe( ).loc[[…]]` 요약 통계 가운데 평균·최솟값·최댓값 줄만 고릅니다.



출력은 평균 6,280W(약 8.4마력)이고 1,148~10,470W에 퍼져 있습니다. 온도 차는 7.6~12.1K로 좁습니다. 숫자가 상식적인 범위인지 이렇게 먼저 확인합니다. 단위를 모르고 토크와 회전수를 그냥 곱했다면 "2π ÷ 60"이 빠진 엉뚱한 숫자가 나왔을 것입니다.

**❸ 설명서의 규칙과 라벨 맞대 보기**

In [ ]:
rpm = df["Rotational speed [rpm]"]
limit = df["Type"].map({"L": 11000, "M": 12000, "H": 13000})
rules = {"HDF": (df.dT < 8.6) & (rpm < 1380),
         "PWF": (df.power < 3500) | (df.power > 9000),
         "OSF": df.strain > limit}
for m, rule in rules.items():
    print(f"{m}: 규칙 {rule.sum()}건, 라벨 {df[m].sum()}건,"
          f" 겹침 {(rule & (df[m] == 1)).sum()}건")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `df["Type"].map({…})` 기계 종류(L·M·H)마다 과부하 기준이 다릅니다. 줄마다 그 기계의 기준값을 붙입니다.
>
> `rules = {…}` 설명서에 적힌 세 규칙을 참·거짓 열로 옮겼습니다. `&`는 그리고, `|`는 또는입니다. 조건마다 괄호를 칩니다.
>
> `df.dT` 열 이름에 빈칸이 없으면 `df["dT"]` 대신 점으로도 씁니다.
>
> `rule.sum( )`, `df[m].sum( )`, `(rule & (df[m] == 1)).sum( )` 규칙에 맞는 작업, 그 유형으로 적힌 고장, 둘 다인 작업의 수입니다.



세 유형 모두 세 숫자가 똑같습니다. 규칙에 맞으면 반드시 고장이고, 고장이면 반드시 규칙에 맞습니다. 실제 공장이라면 센서 잡음과 우연 때문에 이럴 수 없습니다. 이 데이터는 규칙과 난수로 만든 합성 데이터입니다. 방법을 연습하기에는 좋지만 "공장에서는 이렇다"고 말해서는 안 됩니다.

**❹ 학습·시험 나누기: 드문 고장은 비율을 지키며**

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
num = ["Air temperature [K]", "Process temperature [K]",
       "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
df["TypeN"] = df["Type"].map({"L": 0, "M": 1, "H": 2})
base, phys = num + ["TypeN"], num + ["TypeN", "dT", "power", "strain"]
y = df["Machine failure"].values
i_tr, i_te = train_test_split(np.arange(len(df)), test_size=0.2,
                              random_state=42, stratify=y)
print("시험 2,000건 가운데 고장:", y[i_te].sum(), "건, 모두 정상으로 찍으면",
      round((y[i_te] == 0).mean() * 100, 1), "%")

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `num` 센서값 다섯 열입니다. 고장 유형 열(HDF 등)은 정답을 그대로 알려 주므로 특징에 넣지 않습니다.
>
> `df["TypeN"] = …map(…)` 글자 L·M·H를 0·1·2로 바꿉니다(아래 [잠깐만요] 범주를 숫자로).
>
> `base`, `phys` 견줄 두 특징 묶음입니다. `phys`는 `base`에 ②의 세 열을 더한 것입니다.
>
> `train_test_split(np.arange(len(df)), …)` 줄 번호 0~9,999를 나눕니다. 번호로 나누면 같은 번호로 여러 열을 골라 쓸 수 있습니다. 이 데이터는 작업 하나가 한 줄이고 같은 주인이 없으니 무작위로 나누어도 됩니다.
>
> `stratify=y` 학습과 시험의 고장 비율을 3.39%로 같게 맞춥니다.



아무것도 배우지 않고 "모두 정상"이라고만 해도 정확도가 96.6%입니다. 그러니 정확도로는 모델을 견줄 수 없습니다. 고장 68건을 몇 건 찾았나(재현율), 경보를 울리면 몇 번이 진짜인가(정밀도)를 봅니다.

**❺ 센서값 그대로 vs 물리 특징 추가**

In [ ]:
for name, cols in [("센서값 그대로", base), ("물리 특징 추가", phys)]:
    m = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    m.fit(df.loc[i_tr, cols], y[i_tr])
    pr = m.predict_proba(df.loc[i_te, cols])[:, 1]   # 고장일 확률
    p = (pr >= 0.5).astype(int)
    print(f"{name}: 정밀도 {precision_score(y[i_te], p):.3f}"
          f" 재현율 {recall_score(y[i_te], p):.3f}"
          f" F1 {f1_score(y[i_te], p):.3f}, 찾은 고장 {(p & y[i_te]).sum()}건")
imp = pd.Series(m.feature_importances_, index=phys).sort_values()
print(imp[::-1].round(3).to_string())

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `for name, cols in […]` 모델 설정은 같게 두고 특징 묶음만 바꿔 두 번 돌립니다.
>
> `df.loc[i_tr, cols]` 줄 번호와 열 이름으로 고르기입니다. `.loc`은 이름(라벨), `.iloc`은 순서 번호로 고릅니다.
>
> `predict_proba( )[:, 1]` 작업마다 [정상일 확률, 고장일 확률]을 돌려줍니다. `[:, 1]`로 고장일 확률만 꺼냅니다(아래 [잠깐만요] 상자).
>
> `(pr >= 0.5).astype(int)` 확률이 0.5 이상이면 1(경보)로 바꿉니다. 0.5가 `predict( )`의 기본 문턱입니다.
>
> `precision_score`, `recall_score`, `f1_score` 정답과 예측을 주면 세 점수를 계산합니다(3.7의 [잠깐만]).
>
> `m.feature_importances_` 루프가 끝난 뒤의 `m`은 물리 특징 모델입니다. 특징마다 중요도(합 1)를 꺼내 큰 순으로 봅니다.



*(그래프: 물리 특징 모델의 특징 중요도. 빨강은 새로 만든 물리 특징이다. 공구 부담(0.220)과 출력(0.210)이 가장 크고, 온도 차도 원래 온도 두 열보다 크다.)*

같은 모델에 열 세 개를 더했을 뿐인데, 찾은 고장이 42건에서 55건으로 늘고 F1이 0.724에서 0.880으로 올랐습니다. 중요도 1, 2위가 새로 만든 공구 부담과 출력입니다. 다만 이 데이터의 고장은 바로 이 물리 규칙으로 만들어졌으니, 정답을 만든 공식을 특징으로 준 셈입니다. 실제 공장에서도 단위를 아는 특징은 도움이 되지만, 이렇게 깨끗하게 오르리라는 보장은 없습니다.

> **[잠깐만요] 확률을 내는 모델과 문턱값**
>
> 분류 모델 대부분은 답 대신 "고장일 확률 0.37"처럼 확률을 낼 수 있습니다. `predict( )`는 그 확률이 0.5 이상이면 1이라고 답하는 것일 뿐입니다. 경보를 울릴 문턱을 0.3으로 낮추면 놓치는 고장은 줄고 헛경보는 늘며, 0.7로 높이면 반대가 됩니다.
>
> 병원의 검사로 치면, 암 검진은 놓치면 안 되니 문턱을 낮게 두고(재현율 우선), 스팸 필터는 중요한 메일을 버리면 안 되니 문턱을 높게 둡니다(정밀도 우선). 문턱은 통계가 아니라 "틀리면 무엇을 잃나"가 정합니다. 그 값을 데이터로 고르는 방법이 코랩 실습 13-9입니다.

**[코랩 실습 13-6 ❻] 출력 구간별 고장률**

> 📒 **코랩 실습**: 책에는 문제만 있습니다. 결과 그래프는 책 3.7과 같습니다.

In [ ]:
pb = pd.cut(df.power, [0, 3500, 5000, 7000, 9000, 20000])
rate = df.groupby(pb, observed=True)["Machine failure"].agg(["size", "mean"])
print(rate.round(3).to_string())

(rate["mean"] * 100).plot.bar(color=["#C8234B", "#9a9a9a", "#9a9a9a",
                                     "#9a9a9a", "#C8234B"], figsize=(6, 3), rot=0)
plt.xticks(range(5), ["3,500 미만", "3,500~5,000", "5,000~7,000",
                      "7,000~9,000", "9,000 초과"], fontsize=10)
plt.xlabel("기계적 출력 (W)")
plt.ylabel("고장률 (%)")
plt.show()

**코드와 결과 읽기**

> **[코드 읽기] 한 줄씩**
>
> `pd.cut(열, 경계)` 연속된 숫자를 구간으로 나눕니다. 경계가 6개면 구간은 5개입니다(아래 [잠깐만요] 상자).
>
> `df.groupby(pb)["Machine failure"]` 구간마다 작업을 묶고 고장 열만 봅니다. `observed=True`는 작업이 실제로 있는 구간만 남깁니다.
>
> `.agg(["size", "mean"])` 묶음마다 개수와 평균을 한 번에 구합니다. 0과 1의 평균이 곧 고장률입니다.
>
> `(rate["mean"] * 100).plot.bar(…)` 고장률을 %로 바꿔 막대로 그립니다. 양 끝 구간만 빨강으로 칠했습니다.



*(그래프: 기계적 출력 구간별 고장률. 3,500W 미만(31건)과 9,000W 초과(64건)는 모두 고장이었고, 그 사이는 0.8~6.5%였다.)*

3,500W 미만 31건과 9,000W 초과 64건, 모두 95건이 100% 고장입니다. ③에서 본 전력 이상 규칙(95건) 그대로입니다. 가운데 구간에서도 출력이 클수록 고장률이 오릅니다(0.8% → 1.2% → 6.5%). 원래 열(토크, 회전수)만 보면 보이지 않던 경계가 단위를 아는 특징 하나로 드러납니다.

> **[잠깐만요] 구간으로 나누고 묶어서 세기: `pd.cut`과 `groupby`**
>
> 나이를 20대·30대·40대로 묶어 구매율을 보거나, 기온을 구간으로 나누어 전력 수요를 보는 일은 데이터 분석에서 가장 흔한 작업입니다. `pd.cut(열, 경계)`는 숫자를 구간 이름으로 바꾸고, `groupby(구간)`은 같은 구간끼리 묶으며, `.agg([…])`는 묶음마다 개수·평균·합계 등을 구합니다. 4장의 그룹별 요약과 같은 도구입니다.
>
> 구간 경계는 의미가 있는 값으로 고릅니다. 여기서는 설명서의 3,500W와 9,000W를 경계에 넣었습니다. 구간마다 개수(size)도 함께 보아야 합니다. 31건뿐인 구간의 100%는 6,359건 구간의 1.2%보다 훨씬 불안한 숫자입니다.

> **[잠깐만요] 글자 범주를 숫자로 바꾸기: 레이블 인코딩과 원-핫 인코딩**
>
> 모델은 "L", "M", "H" 같은 글자를 계산하지 못합니다. 기계 종류처럼 순서가 있는 범주(낮음 < 중간 < 높음)는 0, 1, 2로 바꾸면 됩니다. 이것이 **레이블 인코딩**이고, `map({"L": 0, "M": 1, "H": 2})`가 그 코드입니다.
>
> 지역(서울, 부산, 광주)이나 색깔처럼 순서가 없는 범주를 0, 1, 2로 바꾸면 모델이 "광주는 서울의 두 배"처럼 없는 순서를 배울 수 있습니다. 이때는 범주마다 0/1 열을 하나씩 만드는 **원-핫 인코딩**을 씁니다. `pd.get_dummies(df["지역"])`이면 서울·부산·광주 열 세 개가 생기고, 그 줄의 지역 칸만 1입니다. 나무 모델은 레이블 인코딩에도 잘 견디지만, 선형 회귀·로지스틱 회귀(8장, 9장)에는 원-핫 인코딩이 안전합니다.

> **[잠깐만요] 특징 중요도: 모델은 어디를 보았나**
>
> 랜덤 포레스트의 **특징 중요도**(feature importance)는 나무들이 그 특징으로 질문했을 때 답이 얼마나 깔끔하게 갈렸는지를 모두 더해, 전체가 1이 되게 나눈 값입니다. 큰 특징일수록 모델이 많이 기대었다는 뜻입니다.
>
> 두 가지를 조심합니다. 첫째, 중요도는 "모델이 많이 썼다"는 뜻이지 "그것이 원인이다"는 뜻이 아닙니다. 둘째, 비슷한 특징이 둘이면(토크와 출력처럼) 중요도를 나눠 가져 둘 다 작아 보일 수 있습니다. 대출 심사, 이탈 예측처럼 이유를 설명해야 하는 일에서는 특징 하나를 뒤섞었을 때 점수가 얼마나 떨어지는지 재는 **순열 중요도**(`sklearn.inspection.permutation_importance`)로 한 번 더 확인합니다.

## 코랩 실습 13-7~13-11 (책에는 없고 이 노트북에만 있습니다)

#### [코랩 실습 13-7] 기울기 각도를 특징으로 더하기

13-4 ④에서 가장 많이 헷갈린 것은 앉기와 서기였습니다. 앉으면 허리가 조금 뒤로 기우니, (식 13-2)로 중력과 세 축의 각도를 구해 특징 110개에 더하고 정확도와 앉기·서기 혼동이 어떻게 바뀌는지 보시오.

**❶ 동작별 기울기 각도**

In [ ]:
def angle(sig):                            # (식 13-2) 중력과 각 축의 각도
    g = sig[:, :, 6:9].mean(axis=1)        # 윈도우마다 축 평균 = 중력 방향
    g = g / np.linalg.norm(g, axis=1, keepdims=True)
    return np.degrees(np.arccos(np.clip(g, -1, 1)))

ang = angle(sig_all)
print(pd.DataFrame([ang[y_all == i].mean(axis=0) for i in (3, 4, 5)],
                   index=ACT[3:], columns=["x축", "y축", "z축"]).round(1))

**코드와 결과 읽기**

- `sig[:, :, 6:9].mean(axis=1)`: 윈도우마다 전체 가속도의 시간 평균입니다. 흔들림이 평균으로 지워지고 중력 방향만 남습니다. 모양은 (윈도우, 3)입니다.
- `g / np.linalg.norm(…, keepdims=True)`: 크기(약 1g)로 나누어 길이 1인 방향으로 만듭니다. 이 값이 곧 (식 13-2)의 aₓ ÷ |a|입니다.
- `np.clip(g, -1, 1)`: 계산 오차로 1.0000001처럼 1을 살짝 넘으면 arccos이 오류를 내므로 -1~1로 자릅니다.
- `np.arccos( )`는 라디안으로 각도를 돌려주고, `np.degrees( )`가 도(°)로 바꿉니다.



서 있을 때 x축과 중력의 각도는 12.9°, 앉으면 16.9°로 4°쯤 커집니다. 눕기는 86.0°로 거의 직각입니다. y축 각도는 앉기 82.5°, 서기 99.2°로 16.7° 차이가 납니다. 이 정도면 앉기와 서기를 가를 수 있을 것 같습니다.

**❷ 각도 3개를 더해 다시 시험**

In [ ]:
G_tr = np.c_[F_tr, angle(sig_tr)]          # 특징 110개 + 각도 3개
G_te = np.c_[F_te, angle(sig_te)]
p_g = RF().fit(G_tr, y_tr).predict(G_te)
cm_g = confusion_matrix(y_te, p_g)
print("각도 3개를 더한 정확도:", round((p_g == y_te).mean() * 100, 2), "%")
print("앉기·서기 혼동:", cm[3, 4] + cm[4, 3], "→",
      cm_g[3, 4] + cm_g[4, 3], "건")

**코드와 결과 읽기**

- `np.c_[A, B]`: 두 표를 옆으로(열 방향으로) 잇습니다. (윈도우, 110)에 (윈도우, 3)을 붙여 (윈도우, 113)이 됩니다. 앞에서 쓴 `np.r_`은 아래로(행 방향으로) 잇습니다.
- `RF( )`: 13-5에서 만든, 같은 설정의 랜덤 포레스트입니다. 특징만 바꾸고 나머지는 13-4와 똑같이 둡니다.
- `cm[3, 4] + cm[4, 3]`: 앉기→서기와 서기→앉기 착각을 더한 수입니다. 13-4의 `cm`과 새 `cm_g`를 견줍니다.



기대와 달리 정확도는 90.23%에서 89.48%로 조금 내려갔고, 앉기·서기 혼동은 오히려 12건 늘었습니다. 각도는 축별 평균으로 계산하는데, 110개 특징에 이미 축별 평균이 들어 있었기 때문입니다. 새 정보가 아닌 특징을 더하면 나무가 질문할 후보만 늘어 흔들릴 수 있습니다.

> **[더 알아보기] 그럴듯한 특징이 늘 도움이 되지는 않는다**
>
> 앉으면 허리가 뒤로 기울어 각도가 커진다는 생각은 물리적으로 옳았고, ①의 숫자도 그것을 보여 주었습니다. 그런데도 모델은 나아지지 않았습니다. 그럴듯한 직관도 실험으로 확인하기 전에는 가설입니다. 특징을 더할 때는 늘 (1) 더하기 전과 후를 같은 시험으로 견주고, (2) 그 특징이 이미 있는 정보를 다르게 쓴 것뿐인지 따져 봅니다. 앉기와 서기를 정말 가르려면 허벅지에 찬 센서처럼 새로운 정보가 필요합니다.

#### [코랩 실습 13-8] 연구팀의 특징 561개와 견주기

연구팀이 공개한 특징 561개를 내려받아, 로지스틱 회귀(9장)와 랜덤 포레스트로 우리의 특징 110개와 견주시오.

**❶ 특징 561개 내려받기**

In [ ]:
for s in ["train", "test"]:                # 연구팀의 특징 561개(약 92MB)
    f = f"{HAR}/{s}/X_{s}.txt"
    if not os.path.exists(f):
        urllib.request.urlretrieve(MIRROR + f"{s}/X_{s}.txt", f)
X561_tr = np.loadtxt(f"{HAR}/train/X_train.txt")
X561_te = np.loadtxt(f"{HAR}/test/X_test.txt")
print("연구팀 특징:", X561_tr.shape, X561_te.shape)

**코드와 결과 읽기**

- 13-1 ①과 같은 방식으로, 없을 때만 내려받습니다. 두 파일이 모두 92MB쯤이라 1~2분 걸립니다.
- `np.loadtxt( )`: 한 줄이 윈도우 하나, 숫자 561개입니다. 줄 순서가 13-1의 `y_tr`, `y_te`와 같으므로 그 정답을 그대로 씁니다.



연구팀은 시간 영역 통계, 주파수 영역 통계, 축 사이의 상관, 각도 등을 모두 모아 561개를 만들었습니다. 우리의 110개보다 다섯 배 많습니다.

**❷ 두 특징 묶음 × 두 모델**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
for name, Xa, Xb in [("특징 561개", X561_tr, X561_te),
                     ("특징 110개", F_tr, F_te)]:
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    acc_lr = lr.fit(Xa, y_tr).score(Xb, y_te)
    acc_f = RF().fit(Xa, y_tr).score(Xb, y_te)
    print(f"{name}: 로지스틱 회귀 {acc_lr * 100:.2f}%,"
          f" 랜덤 포레스트 {acc_f * 100:.2f}%")

**코드와 결과 읽기**

- `make_pipeline(A, B)`: 전처리 A와 모델 B를 한 덩어리로 묶습니다. `fit( )`하면 A로 눈금을 맞춘 뒤 B를 배우고, `score( )`할 때는 학습 때 정한 눈금을 시험에 그대로 씁니다(아래 [잠깐만요] 상자).
- `StandardScaler( )`: 열마다 평균 0, 표준편차 1로 맞춥니다. 로지스틱 회귀는 열의 크기에 민감하기 때문입니다.
- `LogisticRegression(max_iter=2000)`: 여섯 동작을 고르는 선형 분류 모델입니다. 특징이 많으면 답을 찾는 데 반복이 많이 필요해 반복 한도를 늘렸습니다.
- 두 특징 묶음에 같은 두 모델을 돌려 2 × 2로 견줍니다. 시험은 모두 처음 보는 9명입니다.



특징을 공들여 만든 561개가 두 모델 모두에서 높습니다. 그리고 561개에서는 단순한 로지스틱 회귀(95.59%)가 랜덤 포레스트(92.87%)보다 높습니다. 특징이 좋으면 모델은 단순해도 됩니다. 반대로 특징이 모자라면 복잡한 모델로도 메우기 어렵습니다. 분석 시간의 많은 부분을 특징 만들기에 쓰는 까닭입니다.

> **[잠깐만요] 전처리는 학습 데이터로만 맞춘다: 파이프라인**
>
> 표준화(평균 0, 표준편차 1)의 평균과 표준편차를 시험 데이터까지 합쳐 구하면, 시험의 정보가 학습에 조금 새어 듭니다(데이터 누수). 그래서 눈금은 학습 데이터로만 정하고(`fit`), 같은 눈금을 시험에 적용(`transform`)해야 합니다. `make_pipeline( )`으로 묶어 두면 이 순서가 저절로 지켜집니다.
>
> 결측값 채우기, 범주를 숫자로 바꾸기도 같습니다. 학생 성적을 표준점수로 바꿀 때 기준 집단을 정해 두고 새 학생에게 같은 기준을 쓰는 것과 같습니다. 교차 검증(☆ 13-9의 [잠깐만])에서도 파이프라인을 통째로 넘기면 조각마다 눈금을 새로 정해 줍니다.

> **[잠깐만요] 검증 데이터: 시험 전에 보는 모의고사**
>
> 모델은 "고장일 확률 0.37"처럼 확률을 냅니다. 몇 이상이면 경보를 울릴지 정하는 값이 **문턱값**(threshold)입니다. 문턱값을 낮추면 놓치는 고장은 줄지만 헛경보가 늘어납니다.
>
> 문턱값을 시험 데이터로 고르면 시험 문제를 보고 답을 고친 셈이 됩니다. 그래서 학습 데이터 안에서 일부를 **검증 데이터**(validation data)로 떼어 모의고사처럼 씁니다. 문턱값, 나무의 깊이처럼 사람이 정하는 값은 모두 검증 데이터로 고르고, 시험 데이터는 마지막에 한 번만 씁니다. 집값 예측이든 스팸 거르기든 똑같습니다.

#### [코랩 실습 13-9] 문턱값은 검증용 데이터로 고르기

고장을 놓치는 손해가 헛경보의 열 배라고 합시다. 학습 8,000건을 맞추기용 6,000건과 검증용 2,000건으로 다시 나누어 손해가 가장 작은 문턱값을 고르고, 그 문턱값을 시험 2,000건에 한 번만 쓰시오(아래 [잠깐만요] 검증 데이터).

**❶ 학습 데이터를 다시 나누어 검증용 확률 구하기**

In [ ]:
i_fit, i_val = train_test_split(i_tr, test_size=0.25, random_state=0,
                                stratify=y[i_tr])
print("학습 8,000건 → 맞추기", len(i_fit), "건 + 검증", len(i_val), "건")
mv = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
mv.fit(df.loc[i_fit, phys], y[i_fit])
pv = mv.predict_proba(df.loc[i_val, phys])[:, 1]   # 검증의 고장 확률

**코드와 결과 읽기**

- `train_test_split(i_tr, …)`: 이번에는 전체가 아니라 학습용 줄 번호 `i_tr`만 나눕니다. 시험 2,000건(`i_te`)은 건드리지 않습니다.
- `test_size=0.25`: 8,000건의 25%, 2,000건을 검증용으로 뗍니다. `stratify=y[i_tr]`로 고장 비율을 맞춥니다.
- `mv`: 13-6 ⑤와 같은 설정의 모델을 맞추기용 6,000건으로만 배웁니다.
- `pv`: 검증 2,000건의 "고장일 확률"입니다. 문턱값은 이 확률로 고릅니다.

**❷ 문턱값마다 검증 손해 세기**

In [ ]:
cost = lambda yt, p: 10 * ((p == 0) & (yt == 1)).sum() \
    + ((p == 1) & (yt == 0)).sum()           # 놓침 × 10 + 헛경보
rows = []
for t in [0.1, 0.2, 0.3, 0.5, 0.7]:
    p = (pv >= t).astype(int)
    rows.append([t, p.sum(), (p & (y[i_val] == 1)).sum(), cost(y[i_val], p)])
val = pd.DataFrame(rows, columns=["문턱값", "경보", "맞힌 고장", "손해"])
print(val.to_string(index=False))
best = val.loc[val["손해"].idxmin(), "문턱값"]
print("검증 손해가 가장 작은 문턱값:", best)

ts = np.round(np.arange(0.05, 0.96, 0.05), 2)
plt.plot(ts, [cost(y[i_val], (pv >= t).astype(int)) for t in ts],
         marker="o", color="#C8234B")
plt.xlabel("문턱값 (이 확률 이상이면 경보)")
plt.ylabel("검증 데이터의 손해")
plt.gcf().set_size_inches(6, 3)
plt.show()

**코드와 결과 읽기**

- `cost`: 놓친 고장(예측 0, 정답 1)은 10점, 헛경보(예측 1, 정답 0)는 1점으로 센 손해입니다. 줄 끝의 `\`는 다음 줄로 이어 쓴다는 표시입니다.
- `p.sum( )`: 울린 경보 수, `(p & (y[i_val] == 1)).sum( )`: 그 가운데 진짜 고장 수입니다.
- `val["손해"].idxmin( )`: 손해가 가장 작은 줄의 번호이고, `.loc[번호, "문턱값"]`으로 그 문턱값을 꺼냅니다.
- `np.arange(0.05, 0.96, 0.05)`: 0.05부터 0.95까지 0.05 간격의 문턱값 19개로 손해 곡선을 그립니다.



*(그래프: 문턱값에 따른 검증 데이터의 손해(놓침 × 10 + 헛경보). 문턱을 낮추면 헛경보가, 높이면 놓친 고장이 늘어 손해가 U자를 그린다. 0.25~0.3 근처가 가장 작다.)*

문턱을 0.1까지 낮추면 고장 63건을 찾지만 경보가 119번 울려 헛경보가 56번입니다. 0.7로 높이면 헛경보는 1번뿐이지만 고장을 13건 놓쳐 손해가 131입니다. 다섯 후보 가운데는 0.3이 가장 작습니다(75). 그래프의 촘촘한 곡선에서도 바닥은 0.25~0.3입니다. 기본값 0.5는 "놓침과 헛경보의 값이 같다"고 가정한 문턱이라, 놓침이 열 배 비싼 이 공장에는 맞지 않습니다.

**❸ 고른 문턱값을 시험에 한 번만**

In [ ]:
for t in [best, 0.5]:                      # pr: 13-6-5 물리 특징 모델의 시험 확률
    p = (pr >= t).astype(int)
    print(f"문턱값 {t} → 시험: 경보 {p.sum()}건,"
          f" 맞힌 고장 {(p & (y[i_te] == 1)).sum()}건,"
          f" 재현율 {recall_score(y[i_te], p):.3f}, 손해 {cost(y[i_te], p)}")

**코드와 결과 읽기**

- `pr`: 13-6 ⑤에서 8,000건 전체로 배운 물리 특징 모델이 시험 2,000건에 낸 확률입니다. 루프의 마지막 값이 물리 특징 모델의 것입니다.
- 검증으로 고른 0.3과 기본값 0.5를 같은 시험에 적용해 견줍니다. 시험은 여기서 처음이자 마지막으로 씁니다.



검증으로 고른 0.3이 시험에서도 손해가 조금 작습니다(127 대 132). 차이가 검증에서보다 작은 것은 시험 고장이 68건뿐이라 한두 건에 숫자가 크게 흔들리기 때문입니다. 그래도 "시험을 보고 고른 문턱"이 아니므로 이 숫자는 믿을 수 있습니다. 사람이 정하는 값(문턱, 깊이, 특징 묶음)은 모두 이렇게 검증 데이터로 고릅니다.

> **[잠깐만요] 교차 검증: 검증을 여러 번 돌려 평균 내기**
>
> 검증 데이터를 한 번만 떼면 어떤 줄이 뽑혔느냐에 따라 결과가 흔들립니다. **교차 검증**(cross-validation)은 학습 데이터를 다섯 조각으로 나누어, 조각 하나씩 돌아가며 검증에 쓰고 나머지 넷으로 배운 뒤 점수 다섯 개를 평균합니다. 모든 줄이 한 번씩 검증에 쓰입니다.
>
> 코드는 `cross_val_score(모델, X, y, cv=5)` 한 줄입니다. 한 주인이 여러 줄을 남기는 데이터는 `GroupKFold`에 사람 번호를 함께 주고, 시간 순서가 있는 데이터(8장, 12장)는 과거로 배우고 미래로 검증하는 `TimeSeriesSplit`을 씁니다. 나누는 방법은 늘 "실제로 쓰일 때와 똑같이"가 기준입니다.

> **[잠깐만요] 이상 탐지와 고립 숲**
>
> 정답(고장 기록)이 없을 때는 다른 것들과 동떨어진 기록을 찾는 **이상 탐지**(anomaly detection)를 씁니다. 카드 회사가 평소와 다른 결제를 잡아내는 것도 이상 탐지입니다. **고립 숲**(isolation forest)은 데이터를 무작위 질문으로 계속 쪼개, 빨리 혼자 떨어지는 점일수록 이상하다고 봅니다¹⁰⁾.

#### [코랩 실습 13-10] 정답 없이 이상한 작업 찾기

고장 라벨을 쓰지 않고 고립 숲으로 "다른 작업과 동떨어진" 3.39%를 골라, 그 가운데 실제 고장이 몇 건인지, 어떤 고장 유형을 잘 찾는지 보시오(아래 [잠깐만요] 이상 탐지).

**❶ 고립 숲으로 이상한 작업 고르기**

In [ ]:
from sklearn.ensemble import IsolationForest
Xn = StandardScaler().fit_transform(df[num])   # 단위가 다른 열을 같은 눈금으로
iso = IsolationForest(n_estimators=300, contamination=0.0339,
                      random_state=42)
flag = iso.fit_predict(Xn) == -1           # -1 = 이상, 정답은 쓰지 않는다
fail = df["Machine failure"] == 1
print("이상으로 고른 것:", flag.sum(), "건, 그 가운데 실제 고장:",
      (flag & fail).sum(), "건")
print("정밀도:", round((flag & fail).sum() / flag.sum() * 100, 1),
      "%  (전체 고장 비율 3.39%)")

**코드와 결과 읽기**

- `StandardScaler( ).fit_transform(df[num])`: 온도(약 300K), 회전수(약 1,500rpm), 마모(0~250분)처럼 크기가 다른 센서 다섯 열을 평균 0, 표준편차 1로 맞춥니다. 그러지 않으면 숫자가 큰 회전수가 "동떨어짐"을 혼자 정합니다.
- `IsolationForest(…, contamination=0.0339)`: 나무 300그루로 데이터를 무작위로 쪼갭니다. `contamination`은 "이상한 것이 3.39%쯤 있다"고 알려 주는 비율로, 몇 건을 고를지 정합니다.
- `iso.fit_predict(Xn)`: 라벨 없이 배우고 바로 답합니다. 정상은 1, 이상은 -1이라 `== -1`로 참·거짓을 만듭니다.
- `(flag & fail).sum( ) / flag.sum( )`: 고른 것 가운데 실제 고장의 비율, 곧 정밀도입니다. 라벨은 여기서 채점에만 씁니다.



아무렇게나 339건을 고르면 고장은 11건쯤(3.39%)일 텐데, 54건(15.9%)이 나왔습니다. 정답 없이 4.7배를 해낸 것입니다. 하지만 정답으로 배운 13-6의 모델(정밀도 0.965)과는 비교가 되지 않습니다.

**❷ 어떤 고장을 찾았나: 유형별로, 점수 분포로**

In [ ]:
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print({m: f"{(flag & (df[m] == 1)).sum()}/{df[m].sum()}" for m in modes})
sc = -iso.score_samples(Xn)                # 클수록 이상
print("이상 점수 중앙값 정상/고장:", np.median(sc[~fail]).round(3),
      np.median(sc[fail]).round(3))

bins = np.linspace(sc.min(), sc.max(), 40)
plt.hist(sc[~fail], bins=bins, color="#9a9a9a", density=True, label="정상")
plt.hist(sc[fail], bins=bins, color="#C8234B", alpha=0.7, density=True,
         label="실제 고장")
plt.axvline(sc[flag].min(), color="black", linestyle="--")
plt.xlabel("이상 점수 (오른쪽일수록 다른 작업과 동떨어짐)")
plt.ylabel("비율")
plt.legend()
plt.gcf().set_size_inches(6, 3)
plt.show()

**코드와 결과 읽기**

- `modes`: 공구 마모(TWF), 열 방출(HDF), 전력(PWF), 과부하(OSF), 무작위(RNF) 다섯 고장 유형입니다. 유형마다 "찾은 수/전체 수"를 만듭니다.
- `-iso.score_samples(Xn)`: 고립 숲의 점수는 작을수록 이상이라, 부호를 바꿔 "클수록 이상"으로 읽기 쉽게 합니다.
- `~fail`: 물결표(~)는 참·거짓을 뒤집습니다. 정상 작업입니다.
- `plt.hist(…, density=True)`: 정상(9,661건)과 고장(339건)의 개수가 크게 달라, 개수 대신 비율로 그려 모양을 견줍니다. `alpha=0.7`은 반투명입니다.
- `plt.axvline(sc[flag].min( ))`: 이상으로 고른 것 가운데 가장 낮은 점수, 곧 경계선에 세로 점선을 긋습니다.



*(그래프: 고립 숲의 이상 점수. 실제 고장(빨강)은 정상 작업(회색)보다 오른쪽에 몰려 있지만 많이 겹친다. 파선 오른쪽 3.39%를 이상으로 골랐다.)*

출력이 극단으로 치우치는 전력 이상(39/95)은 잘 찾았지만, 값 하나하나는 평범하고 조합이 문제인 열 방출 실패(5/115)는 거의 못 찾았습니다. 무작위 고장(0/19)은 센서에 흔적이 없으니 찾을 수 없는 것이 당연합니다. 이상 탐지는 "평소와 다르다"를 찾을 뿐 "고장이다"를 찾지는 않습니다. 정답이 쌓이기 전 첫 경보 장치로 쓰고, 정답이 모이면 13-6처럼 지도 학습으로 넘어갑니다.

#### [코랩 실습 13-11] 모델의 크기 재기

특징 110개를 받아 64개, 32개 뉴런을 거쳐 여섯 동작을 내는 신경망을 배우고 매개변수를 센 뒤, 깊이 3 나무·신경망·랜덤 포레스트의 정확도와 저장 크기를 견주시오(04절 4.4, 06절 6.1).

**❶ 신경망 배우고 매개변수 세기**

In [ ]:
import pickle
from sklearn.neural_network import MLPClassifier
mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600,
                                  early_stopping=True, random_state=0))
mlp.fit(F_tr, y_tr)
net = mlp[-1]                              # 파이프라인의 마지막 = 신경망
print("가중치 모양:", [w.shape for w in net.coefs_])
n_par = sum(w.size for w in net.coefs_) + sum(b.size for b in net.intercepts_)
print("매개변수:", n_par, "개 → 4바이트면", round(n_par * 4 / 1024, 1), "KB")

**코드와 결과 읽기**

- `MLPClassifier(hidden_layer_sizes=(64, 32))`: 가운데 층(은닉층)이 둘인 신경망입니다. 첫 층 뉴런 64개, 둘째 층 32개입니다(아래 [잠깐만요] 상자).
- `early_stopping=True`: 학습 데이터의 10%를 검증용으로 떼어, 검증 점수가 더 오르지 않으면 배우기를 멈춥니다. 과대적합을 막는 장치입니다. `max_iter=600`은 반복 한도입니다.
- `make_pipeline(StandardScaler( ), …)`: 신경망은 열의 크기에 민감해 표준화를 함께 묶었습니다(☆ 13-8의 [잠깐만]).
- `mlp[-1]`: 파이프라인도 목록처럼 번호로 꺼낼 수 있습니다. 마지막 단계가 신경망입니다.
- `net.coefs_`, `net.intercepts_`: 층과 층 사이의 가중치 표들과 뉴런마다의 편향입니다. `.size`(숫자 개수)를 모두 더하면 매개변수 수입니다.
- `n_par * 4 / 1024`: 실수 하나에 4바이트, 1KB는 1,024바이트입니다(식 13-6).



가중치 표는 110 × 64, 64 × 32, 32 × 6이고, 여기에 편향 64 + 32 + 6개를 더하면 9,382개입니다. 4.4에서 손으로 센 값과 같습니다. 숫자 하나를 1바이트 정수로 줄이면(양자화) 약 9.2KB라 작은 칩에도 들어갑니다.

**❷ 세 모델의 정확도와 저장 크기**

In [ ]:
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(F_tr, y_tr)
for name, model in [("깊이 3 나무", t3), ("신경망 64-32", mlp),
                    ("랜덤 포레스트 200그루", rf)]:
    kb = len(pickle.dumps(model)) / 1024   # 저장 파일의 크기
    print(f"{name}: 정확도 {model.score(F_te, y_te) * 100:.2f}%,"
          f" 저장 파일 {kb:,.1f}KB")

**코드와 결과 읽기**

- `pickle.dumps(모델)`: 파이썬 객체를 파일에 쓸 수 있는 바이트 묶음으로 바꿉니다. 그 길이가 곧 저장 파일의 크기입니다. 실제로 저장하려면 `pickle.dump(모델, open("model.pkl", "wb"))`라고 씁니다.
- `rf`: 13-4 ①의 랜덤 포레스트입니다. 세 모델 모두 같은 특징, 같은 시험으로 견줍니다.
- `{kb:,.1f}`: 천 단위 쉼표와 소수 첫째 자리까지로 숫자를 씁니다.



랜덤 포레스트는 가장 정확하지만 나무 200그루의 마디를 모두 저장해 11MB가 넘습니다. 신경망은 1.4%p 낮은 대신 매개변수만 넣으면 36.6KB입니다(저장 파일이 더 큰 것은 학습 기록 같은 정보가 덧붙기 때문입니다). 깊이 3 나무는 2.1KB지만 71.67%입니다. 어느 것을 고를지는 기기의 메모리와 배터리, 틀렸을 때의 손해가 정합니다(6.1 엣지 AI).

> **[잠깐만요] 신경망: 가중합을 층층이 쌓은 모델**
>
> 뉴런 하나는 들어온 숫자들에 가중치를 곱해 더하고(가중합) 편향을 더한 뒤, 음수면 0으로 바꾸는 것처럼 휘게 하는 함수(활성화 함수)를 지나 다음 층으로 보냅니다. 로지스틱 회귀(9장)가 뉴런 하나라면, 신경망은 그런 뉴런을 층층이 쌓은 것입니다. 가운데 층이 특징을 다시 조합해 새 특징을 만들기 때문에 복잡한 경계도 그릴 수 있습니다.
>
> 학습은 맞히는 방향으로 가중치를 조금씩 고치는 일을 수백 번 되풀이하는 것입니다. 그래서 숫자의 크기를 맞추는 표준화가 중요하고, 반복을 언제 멈출지(`early_stopping`)도 정해야 합니다. 사진·소리·말처럼 특징을 사람이 만들기 어려운 데이터에서 특히 강하며, 11장의 언어 모델도 이 뉴런을 수십억 개 쌓은 것입니다.

## 04절 계산을 파이썬으로 확인하기

책 04절의 손계산(식 13-1~13-6)을 파이썬으로 다시 해 봅니다.

### 4.1 벡터의 크기와 기울기 각도

In [ ]:
import numpy as np
a = np.array([0, 0.6, 0.8])               # 가만히 놓인 휴대폰의 가속도 (g)
size = np.linalg.norm(a)                   # (식 13-1)
print("크기:", round(size, 3), "g")
print("각도(°):", np.degrees(np.arccos(a / size)).round(1))   # (식 13-2)

### [예제 13-1] 걸음 리듬에서 1분 걸음 수 구하기

In [ ]:
fs, N = 50, 128
freq = np.fft.rfftfreq(N, 1 / fs)          # 칸마다의 주파수
print("가장 높은 주파수:", fs / 2, "Hz,  칸 간격:", fs / N, "Hz")
print("4~6번째 칸:", freq[4:7].round(3), "Hz")
print("5번째 칸 → 1분에", round(freq[5] * 60), "걸음")

### [예제 13-2] 밀링 작업의 출력 구하기

In [ ]:
for torque, rpm in [(40, 1500), (60, 1450)]:
    P = torque * 2 * np.pi * rpm / 60      # (식 13-5)
    print(f"토크 {torque}Nm, {rpm}rpm → {P:,.0f}W")

### 4.4 모델의 크기: 매개변수와 바이트

In [ ]:
layers = [110, 64, 32, 6]                  # 입력 110개 → 64 → 32 → 6동작
n = sum(a * b + b for a, b in zip(layers[:-1], layers[1:]))
print("매개변수:", f"{n:,}", "개")
print("4바이트 실수:", round(n * 4 / 1024, 1), "KB,  1바이트 정수:", round(n / 1024, 1), "KB")

## [미니 프로젝트 13] 종합 프로젝트: 우리 도시의 센서로 질문 하나에 답하기


**질문 고르기.** 서울에는 기상 관측소, 대기 측정소, 지하철 개찰구, 전력 계량기 같은 센서가 흩어져 있습니다. 이들이 남긴 공공데이터로 아래 세 질문 가운데 하나를 골라 답하십시오.

- 주제 A. 기온을 알면 내일 전국의 전력 수요를 얼마나 맞힐 수 있을까? (8장 회귀, 12장 기준선)
- 주제 B. 비가 오는 평일에는 지하철 승객이 줄까? (4장 그림, 5장 검정)
- 주제 C. 오늘의 초미세먼지와 날씨로 내일 "나쁨"을 맞힐 수 있을까? (9장 분류, 10장 나무, 12장 시간 순서)

**데이터.** 모두 공공누리 조건에 따라 출처를 밝히고 쓸 수 있는 한국 공공데이터입니다. 프로젝트에 맞게 줄인 네 파일이 강의 자료에 들어 있습니다. 파일을 코랩 왼쪽 파일 창에 끌어다 놓고 시작합니다. 날씨 파일은 세 주제가 함께 씁니다.

**[표 13-3] 종합 프로젝트 데이터**

| 파일 | 출처 | 크기 | 한 행 |
|---|---|---|---|
| asos_서울108_시간자료_2023_2026.csv | 기상청 종관기상관측(ASOS) 서울 관측소 | 33,241행 × 7열 | 한 시각의 기온, 비, 바람, 습도 |
| kpx_시간별_전국_전력수요량_2021_2025.csv | 한국전력거래소 | 1,827행 × 25열 | 하루, 열은 1~24시의 수요(MW) |
| 서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv | 서울교통공사 | 1,462행 × 22열 | 하루, 승차 또는 하차. 열은 시간대별 인원과 합계(모든 역의 합) |
| 에어코리아_서울_도시대기평균_시간별_2023_2026.csv | 한국환경공단 에어코리아 확정자료 | 29,184행 × 4열 | 한 시각, 서울 도시대기 측정소 25곳의 PM2.5 평균 |

원자료는 코랩에 올리기에 큽니다. 에어코리아 원자료는 서울 측정소 40곳의 시간별 기록 117만 행(약 131MB)이라, 도로변 측정소 15곳을 빼고 도시대기 25곳을 시각마다 평균했습니다. 지하철 원자료는 한 해에 약 20만 행(역마다, 날마다, 승차와 하차)이라, 날짜와 승하차마다 모든 역을 더했습니다. 날씨 파일에는 프로젝트에 쓰는 열만 남겼습니다.

**데이터 파일 올리기.** 프로젝트에 맞게 줄인 아래 네 파일이 강의 자료에 들어 있습니다. 코랩 왼쪽의 📁 파일 창에 끌어다 놓으십시오. 고른 주제의 파일과 날씨 파일만 있어도 됩니다.

- `asos_서울108_시간자료_2023_2026.csv` (모든 주제)
- `kpx_시간별_전국_전력수요량_2021_2025.csv` (주제 A)
- `서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv` (주제 B, 날짜·승하차마다 모든 역을 더한 파일)
- `에어코리아_서울_도시대기평균_시간별_2023_2026.csv` (주제 C)

출처: 기상청 기상자료개방포털, 한국전력거래소·서울교통공사(공공데이터포털), 한국환경공단 에어코리아. 공공누리 조건에 따라 출처를 밝힙니다.

> **[더 알아보기] ☆ 포털에서 직접 받아 합치기**
>
> 전력 수요 파일은 12장 미니 프로젝트와 같은 파일이고, 해마다의 파일을 잇는 방법도 12장과 같습니다. 지하철 원자료는 공공데이터포털에 해마다 올라와 있습니다. 열 이름이 `수송일자`, `승하차구분`, `13-14시간대` 꼴이고 합계 열이 없습니다. 늦은 시간 열은 해에 따라 다르고, 2023년은 기간이 겹치는 파일 두 개(1~9월, 1~12월)로 올라와 있습니다. 그래서 열 이름을 맞추고, 같은 날·같은 역·같은 승하차는 하나만 남긴 뒤 더합니다.

In [ ]:
# ☆ 포털에서 받은 역별 원자료를 서울 전체 하루 단위로 합치기 (강의 자료의 파일을 쓰면 건너뜁니다)
import glob
RAW = sorted(glob.glob("서울교통공사_역별*승하차인원*.csv"))     # 받은 파일 이름에 맞게 고칩니다
if not RAW:
    print("포털에서 받은 파일이 없습니다. 강의 자료의 서울 전체 파일로 하십시오.")
else:
    parts = []
    for f in RAW:
        t = pd.read_csv(f, encoding="cp949")                    # 포털 파일의 한글은 cp949
        t = t.rename(columns={"수송일자": "날짜", "승하차구분": "구분"})
        t.columns = [c.replace("시간대", "시") for c in t.columns]   # 13-14시간대 → 13-14시
        late = [c for c in ["23-24시", "24시이후"] if c in t.columns]   # 늦은 시간 열은 해마다 다르다
        if late:
            t["23시이후"] = t[late].sum(axis=1)
            t = t.drop(columns=late)
        parts.append(t)
    raw = pd.concat(parts).drop_duplicates(["날짜", "역번호", "구분"], keep="last")  # 기간이 겹치는 파일
    hours = [c for c in raw.columns if c.endswith(("시", "이전", "이후"))]
    city = raw.groupby(["날짜", "구분"])[hours].sum()
    city["합계"] = city.sum(axis=1)
    city.reset_index().to_csv("지하철_직접_합친_파일.csv", index=False)
    print(len(RAW), "개 파일 →", city.shape, city.index[0][0], "~", city.index[-1][0])

**1단계: 데이터 신원 확인**(다섯 질문 ①, [파이썬 실습 13-1] 응용). 공통 날씨 파일을 하루 단위로 묶고, 고른 주제의 파일을 읽으시오. 기간, 빈칸, 한 행의 뜻, 이상한 값을 확인하시오.

In [ ]:
# 종합 프로젝트 공통: 기상청 서울(108) 시간자료 → 하루 단위
w = pd.read_csv("asos_서울108_시간자료_2023_2026.csv",
                usecols=["일시", "기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)"])
w.columns = ["일시", "기온", "강수량", "풍속", "습도"]
w["일시"] = pd.to_datetime(w["일시"])
print("시간자료:", len(w), "행,", w["일시"].min(), "~", w["일시"].max())
print("빈칸:", w.isna().sum().to_dict())
w["강수량"] = w["강수량"].fillna(0)          # 비가 안 온 시각은 빈칸으로 적혀 있다
day = w.groupby(w["일시"].dt.normalize()).agg(기온=("기온", "mean"), 강수량=("강수량", "sum"),
                                             풍속=("풍속", "mean"), 습도=("습도", "mean"))
print("하루 단위:", day.shape)

강수량 빈칸 29,375개는 고장이 아니라 "비가 오지 않은 시각"입니다. 기상청은 비가 오지 않으면 칸을 비워 둡니다. 그래서 0으로 채웠습니다. 풍속의 빈칸 68개는 진짜 결측이라 그대로 두었습니다. 같은 빈칸이라도 열마다 뜻이 다릅니다(3장).

In [ ]:
# 주제 A: 전력 수요 (한국전력거래소, 전국, 시간별 MW)
pw = pd.read_csv("kpx_시간별_전국_전력수요량_2021_2025.csv", parse_dates=["날짜"])
hours = [f"{h}시" for h in range(1, 25)]
pw["GWh"] = pw[hours].mean(axis=1) * 24 / 1000        # 하루 전력량
A = pw.set_index("날짜")[["GWh"]].join(day, how="inner")
print(A.shape, A.index.min().date(), "~", A.index.max().date())
print(A.groupby(pd.cut(A["기온"], [-15, 0, 10, 20, 25, 35]), observed=True)["GWh"].mean().round(0))
A.plot.scatter(x="기온", y="GWh", s=6, color="#9a9a9a", figsize=(5, 3.2))
plt.xlabel("서울 하루 평균 기온 (°C)")
plt.ylabel("전국 하루 전력 수요 (GWh)")
plt.show()

In [ ]:
# 주제 B: 지하철 (서울교통공사, 모든 역을 더한 하루·시간대별 승하차)
sub = pd.read_csv("서울교통공사_서울전체_일별_시간대별_승하차_2023_2024.csv", parse_dates=["날짜"])
B = sub[sub["구분"] == "승차"].set_index("날짜")[["13-14시", "합계"]] / 10000   # 만 명
month = B.groupby(B.index.to_period("M")).mean().round(1)
print(month.loc["2024-02":"2024-06"])
month.index = month.index.strftime("%y.%m")          # 23.01 꼴의 글자로
month["13-14시"].plot(marker="o", color="#C8234B", figsize=(6, 3), rot=0)
plt.ylabel("13~14시 승차 (만 명, 하루 평균)")
plt.xlabel("")
plt.show()

주제 B를 고른 사람은 여기서 멈춰야 합니다. 1장, 3장, 5장의 미니 프로젝트에서 만난 그 2024년 4월입니다. 13~14시 승차가 하루 평균 175.9만 명으로, 다른 달(20만 명 남짓)의 여덟 배쯤입니다. 그래서 그달의 하루 합계도 635.2만 명으로 부풀었습니다. 서울교통공사가 공개한 원자료에 그대로 있는 값입니다. 이 칸만 빼고 다시 더할지, 그달을 통째로 뺄지 정하고 그 결정을 기록해 두십시오. 3장은 이 칸을 앞뒤 시간대의 평균으로 채웠고, 5장은 그달을 통째로 뺐습니다.

In [ ]:
# 주제 C: 초미세먼지 (에어코리아 확정자료, 서울 도시대기 측정소 25곳 평균)
air = pd.read_csv("에어코리아_서울_도시대기평균_시간별_2023_2026.csv", parse_dates=["일시"])
air["날"] = (air["일시"] - pd.Timedelta(hours=1)).dt.normalize()   # 24시 기록은 그날 것
C = air.groupby("날")["PM25"].mean().to_frame("PM25").join(day, how="inner")
C["내일나쁨"] = (C["PM25"].shift(-1) > 35).astype(int)            # 36㎍/㎥ 이상이면 "나쁨"
C = C.iloc[:-1]
print(C.shape, " 다음 날이 '나쁨' 이상인 날:", round(C["내일나쁨"].mean() * 100, 1), "%")
print(C.groupby(C.index.month)["PM25"].mean().round(1).to_dict())

**2단계: 그림 한 장**(다섯 질문 ④, 4장). 질문에 답하는 그림 한 장을 그리시오. 그림 하나는 질문 하나에 답해야 합니다.

**3단계: 기준선 만들기**(다섯 질문 ②, 8·9·12장). 주제 A는 "지난주 같은 요일", 주제 C는 "내일도 오늘과 같다"를 기준선으로 쓰시오. 주제 B는 같은 달, 같은 요일의 보통 날과 견주시오. 날씨가 아니라 요일과 계절이 만든 차이를 빼기 위해서입니다.

**4단계: 이 책의 기법 쓰기**(다섯 질문 ③). A는 기온과 휴일로 회귀를(8장), B는 비 온 날과 아닌 날의 차이를 검정으로(5장), C는 로지스틱 회귀나 랜덤 포레스트로 분류를(9·10장) 하시오. A와 C는 2023~2024년으로 배우고 그 뒤로 시험하시오. C의 문턱값은 시험 기간이 아니라 그 앞의 검증 기간으로 고르시오([파이썬 실습 13-9]).

**5단계: 해석과 한계**(다섯 질문 ⑤). [표 13-2]의 다섯 질문에 한 줄씩 답하는 표를 만드시오. 예를 들어 주제 A에서 서울 기온 하나로 전국 수요를 맞히는 것은 무엇을 놓칠까요? 그날의 실제 기온은 전날 밤에는 알 수 없고 예보만 알 수 있다는 점은 어떻게 다룰까요?

**3시간 계획.** 첫 1시간에 1~2단계, 다음 1시간에 3~4단계, 마지막 1시간에 5단계와 제출물 정리를 합니다.

**제출물.** 그림 2장(2단계 그림, 기준선과 모델을 견준 그림), 숫자 표 1개(기준선과 모델의 점수), 결론 5문장 이내, 다섯 질문 표.

**[표 13-4] 종합 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 기간, 빈칸의 뜻, 이상한 값(예: 2024년 4월)을 찾아 처리하고 기록함 | 빈칸만 처리함 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 데이터의 주기에 맞는 기준선과 시간 순서로 견줌 | 기준선은 있으나 무작위로 나눔 | 기준선 없이 점수만 씀 |
| 기법 적용(30%) | 앞 장의 기법을 맞게 쓰고, 문턱값이나 비교 조건을 시험 기간 밖에서 정함 | 기법은 맞으나 시험 기간으로 고름 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 다섯 질문에 모두 답하고, 모델이 기준선을 못 이긴 경우도 그대로 씀 | 한계를 한두 가지만 씀 | 숫자만 나열함 |

### 내 분석

여기부터 고른 주제의 2~5단계를 직접 해 보십시오.

In [ ]:
# 2단계: 질문에 답하는 그림 한 장

In [ ]:
# 3단계: 기준선 만들기

In [ ]:
# 4단계: 이 책의 기법 쓰기 (시간 순서로 나누어 시험하기)

**5단계: 해석과 한계** — 다섯 질문에 한 줄씩 답해 보십시오.

| 질문 | 내 답 |
|---|---|
| ① 누가, 왜, 어떻게 모았나? | |
| ② 가장 단순한 답과 견주었나? | |
| ③ 실제로 쓰일 때처럼 시험했나? | |
| ④ 틀리면 무엇을 잃나? | |
| ⑤ 아직 모르는 것은 무엇인가? | |

## ✏️ [연습문제 13-6] 공구 마모 고장을 규칙으로 찾기

AI4I 데이터에서 공구 마모 고장(TWF)이 일어난 작업의 공구 마모 시간 범위를 구하고, "마모 시간 200분 이상이면 고장 경보"라는 규칙의 정밀도와 재현율을 구하시오. 결과를 보고 이 규칙을 공장에서 쓸지 판단하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오. [파이썬 실습 13-6]의 `df`를 이어서 씁니다.

In [ ]:
twf = df["TWF"] == 1
print("공구 마모 고장(TWF):", twf.sum(), "건, 마모 시간",
      df.loc[twf, "Tool wear [min]"].min(), "~", df.loc[twf, "Tool wear [min]"].max(), "분")
rule = df["Tool wear [min]"] >= 200
print("마모 200분 이상:", rule.sum(), "건, 그 가운데 TWF:", (rule & twf).sum(), "건")
print("정밀도:", round((rule & twf).sum() / rule.sum() * 100, 1), "%, 재현율:",
      round((rule & twf).sum() / twf.sum() * 100, 1), "%")